In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os, glob, random, tarfile
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

from scipy.stats import t

from sklearn.isotonic import IsotonicRegression

import torch.nn.functional as F

from IPython.display import display

from collections import defaultdict

import math
from scipy.stats import norm

import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")
import numpy as np

from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from scipy.optimize import linear_sum_assignment

# **CONFIG**

In [ ]:
SLOT_CKPT = "SlotAttentionCheckpoints/slot_attention_epoch_50.pth"

DATA_TRAIN_TAR = "datasets/trainset.tar"
DATA_TEST_TAR  = "datasets/testset.tar"
EXTRACT_ROOT = "causal3DIdent"

SAVE_DIR = "Experiment_Results"

CKPT_ROOT = os.path.join(SAVE_DIR, "checkpoints")
CKPT_EVERY = 5

LOG_ROOT = os.path.join(SAVE_DIR, "logs")
os.makedirs(LOG_ROOT, exist_ok=True)

SEEDS = [45, 46, 47]
NUM_EPOCHS = 5
BATCH_SIZE = 512
LR = 1e-3

NUM_MATCHED_PAIRS = 500
FACTOR_NAMES = [f"f{k}" for k in range(10)]

# SPLITS
VAL_FRAC = 0.10          # 10% of TRAINSET reserved as "validation pool"
VAL_CAL_FRAC = 0.50      # split validation pool into 50% val_sel and 50% val_cal


DEVICE = "cuda"

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
NUM_WORKERS = 4

In [ ]:
def make_fixed_slot_init_noise(feats, num_slots, seed=1234):
    B = feats.size(0)
    D = feats.size(-1)

    g = torch.Generator(device=feats.device)
    g.manual_seed(int(seed))

    base_noise = torch.randn(
        1,
        num_slots,
        D,
        generator=g,
        device=feats.device,
        dtype=feats.dtype,
    )

    init_noise = base_noise.expand(B, -1, -1).contiguous()
    return init_noise

**Extract dataset**

In [ ]:
def extract_tar_with_progress(tar_path, output_dir):
    print(f"\nExtracting: {tar_path}\n")
    with tarfile.open(tar_path, "r:*") as tar:
        members = tar.getmembers()
        for member in tqdm(members, total=len(members), desc="Extracting", unit="files"):
            tar.extract(member, path=output_dir)
    print("Extraction complete!")

os.makedirs(EXTRACT_ROOT, exist_ok=True)
train_root = os.path.join(EXTRACT_ROOT, "trainset")
test_root  = os.path.join(EXTRACT_ROOT, "testset")

if not os.path.exists(train_root) or len(glob.glob(train_root + "/**/*.png", recursive=True)) == 0:
    extract_tar_with_progress(DATA_TRAIN_TAR, EXTRACT_ROOT)
else:
    print("trainset already extracted, skipping.")

if not os.path.exists(test_root) or len(glob.glob(test_root + "/**/*.png", recursive=True)) == 0:
    extract_tar_with_progress(DATA_TEST_TAR, EXTRACT_ROOT)
else:
    print("testset already extracted, skipping.")

print("Train images:", len(glob.glob(train_root + "/images_*/**/*.png", recursive=True)))
print("Test images :", len(glob.glob(test_root  + "/images_*/**/*.png", recursive=True)))

trainset already extracted, skipping.
testset already extracted, skipping.
Train images: 252000
Test images : 25200


**Dataset / Dataloaders**

In [ ]:
class Causal3DIdentDataset(Dataset):
    def __init__(self, root, image_size=64):
        # Load latent factors
        latent_files = sorted(glob.glob(os.path.join(root, "latents_*.npy")))
        latents_list = [np.load(f) for f in latent_files]
        self.latents = np.concatenate(latents_list, axis=0).astype(np.float32)

        # Load image paths
        self.image_paths = []
        shard_dirs = sorted(glob.glob(os.path.join(root, "images_*")))
        for folder in shard_dirs:
            self.image_paths.extend(
                sorted(glob.glob(os.path.join(folder, "*.png")))
            )

        # alignment check
        assert len(self.image_paths) == len(self.latents), (
            f"Mismatch between images and latents: "
            f"{len(self.image_paths)} images vs {len(self.latents)} latents"
        )

        # Image transforms
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        x = self.transform(img)
        y = torch.from_numpy(self.latents[idx])
        return x, y


full_train_ds = Causal3DIdentDataset(train_root, image_size=64)
test_ds       = Causal3DIdentDataset(test_root, image_size=64)


In [ ]:
print("Latents shape train:", full_train_ds.latents.shape)
print("Latents shape test :", test_ds.latents.shape)

Latents shape train: (252000, 10)
Latents shape test : (25200, 10)


# **OOD DATASET / LOADERS**

In [ ]:
def _severity_value(severity, value_map, name):
    if severity not in value_map:
        raise ValueError(f"Unknown severity={severity} for {name}. Expected one of {list(value_map.keys())}")
    return value_map[severity]

In [ ]:
class Causal3DIdentOODDataset(Dataset):
    """
    Deterministic OOD dataset.
    - gaussian_blur is deterministic.
    - brightness is deterministic, not random ColorJitter.
    - gaussian_noise is deterministic per sample index, so different models see the same corrupted image.
    """
    def __init__(
        self,
        root,
        image_size=64,
        corruption="gaussian_blur",
        severity=3,
        noise_base_seed=12345,
    ):
        latent_files = sorted(glob.glob(os.path.join(root, "latents_*.npy")))
        latents_list = [np.load(f) for f in latent_files]
        self.latents = np.concatenate(latents_list, axis=0).astype(np.float32)

        self.image_paths = []
        shard_dirs = sorted(glob.glob(os.path.join(root, "images_*")))
        for folder in shard_dirs:
            self.image_paths.extend(sorted(glob.glob(os.path.join(folder, "*.png"))))

        assert len(self.image_paths) == len(self.latents), (
            f"Mismatch between images and latents: "
            f"{len(self.image_paths)} images vs {len(self.latents)} latents"
        )

        self.image_size = image_size
        self.corruption = corruption
        self.severity = severity
        self.noise_base_seed = int(noise_base_seed)

        self.base_transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

        self.blur_sigma_map = {
            1: 0.5,
            2: 1.0,
            3: 1.5,
            4: 2.0,
            5: 2.5,
        }

        self.brightness_delta_map = {
            1: 0.15,
            2: 0.30,
            3: 0.45,
            4: 0.60,
            5: 0.75,
        }

        self.noise_std_map = {
            1: 0.03,
            2: 0.06,
            3: 0.09,
            4: 0.12,
            5: 0.15,
        }

        valid_corruptions = ["gaussian_blur", "brightness", "gaussian_noise"]
        if corruption not in valid_corruptions:
            raise ValueError(f"Unknown corruption: {corruption}. Expected one of {valid_corruptions}")

    def __len__(self):
        return len(self.image_paths)

    def _apply_corruption(self, x, idx):
        if self.corruption == "gaussian_blur":
            sigma = _severity_value(self.severity, self.blur_sigma_map, "gaussian_blur")
            return transforms.GaussianBlur(kernel_size=5, sigma=sigma)(x)

        if self.corruption == "brightness":
            # Fixed brightness shift for reproducibility.
            # Severity controls how much brighter the image becomes.
            delta = _severity_value(self.severity, self.brightness_delta_map, "brightness")
            factor = 1.0 + delta
            return torch.clamp(x * factor, 0.0, 1.0)

        if self.corruption == "gaussian_noise":
            std = _severity_value(self.severity, self.noise_std_map, "gaussian_noise")

            # Deterministic noise per image index.
            g = torch.Generator(device="cpu")
            g.manual_seed(self.noise_base_seed + int(idx))

            noise = torch.randn(
                x.shape,
                generator=g,
                dtype=x.dtype,
                device="cpu"
            ) * std

            return torch.clamp(x + noise, 0.0, 1.0)

        raise ValueError(f"Unknown corruption: {self.corruption}")

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        x = self.base_transform(img)
        x = self._apply_corruption(x, idx)
        y = torch.from_numpy(self.latents[idx])
        return x, y


def build_ood_loader(
    root,
    corruption="gaussian_blur",
    severity=3,
    batch_size=512,
    image_size=64,
    num_workers=NUM_WORKERS,
    noise_base_seed=12345,
):
    ds = Causal3DIdentOODDataset(
        root=root,
        image_size=image_size,
        corruption=corruption,
        severity=severity,
        noise_base_seed=noise_base_seed,
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return ds, loader

# **SLOT ATTENTION AE**

In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self, in_channels=3, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 5, padding=2), nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 5, padding=2), nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 5, padding=2), nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 5, padding=2), nn.ReLU(),
        )

    def forward(self, x):
        feats = self.net(x)
        B, C, H, W = feats.shape
        return feats.view(B, C, H * W).permute(0, 2, 1)  # (B, N, C)

In [ ]:
class SlotAttention(nn.Module):
    def __init__(self, num_slots, dim, iters=3, eps=1e-8, hidden_dim=128):
        super().__init__()
        self.num_slots = num_slots
        self.dim = dim
        self.iters = iters
        self.eps = eps
        self.scale = dim ** -0.5

        self.slots_mu = nn.Parameter(torch.randn(1, 1, dim))
        self.slots_log_sigma = nn.Parameter(torch.zeros(1, 1, dim))

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.gru = nn.GRUCell(dim, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, dim)
        )

        self.norm_input = nn.LayerNorm(dim)
        self.norm_slots = nn.LayerNorm(dim)
        self.norm_pre_ff = nn.LayerNorm(dim)

    def forward(self, inputs, init_noise=None):
        """
        init_noise: Tensor shaped (B, K, D) used for slot initialization noise.
        """
        B, N, D = inputs.shape
        K = self.num_slots

        inputs = self.norm_input(inputs)

        mu = self.slots_mu.expand(B, K, -1)
        sigma = torch.exp(self.slots_log_sigma).expand(B, K, -1)  # compute once & expand

        # noise handling (keeps training/inference procedure identical when init_noise=None)
        if init_noise is None:
            init_noise = torch.randn_like(mu)
        else:
            init_noise = init_noise.to(device=mu.device, dtype=mu.dtype)

        slots = mu + sigma * init_noise

        k = self.to_k(inputs)
        v = self.to_v(inputs)

        for _ in range(self.iters):
            slots_prev = slots

            q = self.to_q(self.norm_slots(slots))
            attn_logits = torch.einsum("bid,bjd->bij", q, k) * self.scale
            attn = attn_logits.softmax(dim=1) + self.eps
            attn = attn / attn.sum(dim=-1, keepdim=True)

            updates = torch.einsum("bjd,bij->bid", v, attn)

            slots = self.gru(
                updates.reshape(-1, D),
                slots_prev.reshape(-1, D)
            )
            slots = slots.reshape(B, K, D)
            slots = slots + self.mlp(self.norm_pre_ff(slots))

        return slots


In [ ]:
class SpatialBroadcastDecoder(nn.Module):
    def __init__(self, slot_dim, hidden_dim=64, out_channels=3, resolution=64):
        super().__init__()
        self.resolution = resolution
        self.init_res = 8
        self.fc = nn.Linear(slot_dim, hidden_dim * self.init_res * self.init_res)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, out_channels + 1, 4, stride=2, padding=1),
        )

    def forward(self, slots):
        B, K, D = slots.shape
        H = W = self.resolution
        x = self.fc(slots).view(B * K, -1, self.init_res, self.init_res)
        x = self.dec(x).view(B, K, -1, H, W)
        rgb, alpha = x[:, :, :3], x[:, :, 3:]
        masks = alpha.softmax(dim=1)
        recon = torch.sum(rgb * masks, dim=1)
        return recon, masks


In [ ]:
class SlotAttentionAE(nn.Module):
    def __init__(self, num_slots=4, slot_dim=64, resolution=64):
        super().__init__()
        self.encoder = ConvEncoder(3, 64)
        self.project = nn.Linear(64, slot_dim)
        self.slot_attention = SlotAttention(num_slots, slot_dim, iters=3)
        self.decoder = SpatialBroadcastDecoder(slot_dim, 64, 3, resolution)

    def forward(self, x, init_noise=None):
        feats = self.project(self.encoder(x))
        slots = self.slot_attention(feats, init_noise=init_noise)
        recon, masks = self.decoder(slots)
        return recon, slots, masks

In [ ]:
ckpt_sa = torch.load(SLOT_CKPT, map_location=device)
sa_model = SlotAttentionAE(num_slots=4, slot_dim=64, resolution=64).to(device)
sa_model.load_state_dict(ckpt_sa["model_state_dict"])
sa_model.eval()
for p in sa_model.parameters():
    p.requires_grad = False
print("Loaded SlotAttn epoch:", ckpt_sa.get("epoch", "N/A"))


Loaded SlotAttn epoch: 50


## **Deterministic Slot-Set Extraction + Dataset Wrapper**

In [ ]:
@torch.no_grad()
def slots_from_dataset(sa_model, dataset, batch_size=64, desc="slots", slot_init_seed=1234):
    sa_model.eval()
    sa = sa_model.slot_attention

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    S_list, Y_list = [], []

    for x, y in tqdm(loader, desc=desc):
        x = x.to(device, non_blocking=True)

        feats = sa_model.project(sa_model.encoder(x))
        K = sa.num_slots

        init_noise = make_fixed_slot_init_noise(
            feats=feats,
            num_slots=K,
            seed=slot_init_seed,
        )

        slots = sa(feats, init_noise=init_noise)

        S_list.append(slots.detach().cpu())
        Y_list.append(y.float().cpu())

    return torch.cat(S_list, dim=0), torch.cat(Y_list, dim=0)

In [ ]:
class SlotSetDataset(Dataset):
    def __init__(self, slots_tensor: torch.Tensor, y: torch.Tensor):
        assert slots_tensor.shape[0] == y.shape[0]
        self.S = slots_tensor
        self.Y = y

    def __len__(self):
        return self.S.shape[0]

    def __getitem__(self, idx):
        return self.S[idx], self.Y[idx]


print("\n=== Precomputing deterministic FULL slots ONCE (shared across all seeds) ===")

S_train_all, Y_train_all = slots_from_dataset(
    sa_model, full_train_ds, batch_size=BATCH_SIZE, desc="slots_train_all"
)

S_test_all, Y_test_all = slots_from_dataset(
    sa_model, test_ds, batch_size=BATCH_SIZE, desc="slots_test_all"
)

assert not S_train_all.is_cuda and not S_test_all.is_cuda
print("S_train_all:", S_train_all.shape, "Y_train_all:", Y_train_all.shape)
print("S_test_all :", S_test_all.shape,  "Y_test_all :",  Y_test_all.shape)


=== Precomputing deterministic FULL slots ONCE (shared across all seeds) ===


slots_test_all: 100%|██████████| 50/50 [00:16<00:00,  3.08it/s]

S_train_all: torch.Size([252000, 4, 64]) Y_train_all: torch.Size([252000, 10])
S_test_all : torch.Size([25200, 4, 64]) Y_test_all : torch.Size([25200, 10])


# **PREDICTOR / ARCHITECTURES**

In [ ]:
class MeanPoolLinearProbFactorPredictor(nn.Module):
    """
    Baseline 0:
      raw slots -> mean pooling -> linear head -> (mu, logvar)

     raw mean-pooled slots baseline
     The raw mean pooled slots baseline has no hidden MLP before prediction.
    """
    def __init__(self, slot_dim=64, out_dim=10):
        super().__init__()
        self.out_dim = out_dim
        self.head = nn.Linear(slot_dim, out_dim * 2)

    def forward(self, slots, return_latent=False):
        pooled = slots.mean(dim=1)          # (B, D)
        out = self.head(pooled)

        mu = out[:, :self.out_dim]
        logvar = out[:, self.out_dim:]

        if return_latent:
            # z and set_repr are both the raw mean-pooled slot vector.
            return mu, logvar, pooled, pooled

        return mu, logvar

In [ ]:
class MeanPoolProbFactorPredictor(nn.Module):
    """
    Baseline 1:
      slots -> mean pooling -> MLP -> z -> (mu, logvar)
      mean-pooled MLP baseline
    """
    def __init__(self, slot_dim=64, z_dim=64, hidden_dim=128, out_dim=10, dropout=0.0):
        super().__init__()
        self.out_dim = out_dim
        self.z_dim = z_dim

        self.encoder = nn.Sequential(
            nn.Linear(slot_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, z_dim),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim * 2),
        )

    def forward(self, slots, return_latent=False):
        pooled = slots.mean(dim=1)          # (B, D)
        z = self.encoder(pooled)            # (B, z_dim)
        out = self.head(z)

        mu = out[:, :self.out_dim]
        logvar = out[:, self.out_dim:]

        if return_latent:
            set_repr = pooled
            return mu, logvar, z, set_repr

        return mu, logvar

In [ ]:
class FlatMLPProbFactorPredictor(nn.Module):
    """
    Baseline 2:
      slots -> flatten all slots -> MLP -> z -> (mu, logvar)
      The flattened-slots MLP baseline is order-sensitive, but gives the model access to all slot vectors directly.
    """
    def __init__(self, num_slots=4, slot_dim=64, z_dim=64, hidden_dim=256, out_dim=10, dropout=0.0):
        super().__init__()
        self.out_dim = out_dim
        self.z_dim = z_dim
        self.num_slots = num_slots
        self.slot_dim = slot_dim

        in_dim = num_slots * slot_dim

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, z_dim),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim * 2),
        )

    def forward(self, slots, return_latent=False):
        B = slots.shape[0]
        flat = slots.reshape(B, -1)          # (B, K*D)
        z = self.encoder(flat)
        out = self.head(z)

        mu = out[:, :self.out_dim]
        logvar = out[:, self.out_dim:]

        if return_latent:
            set_repr = flat
            return mu, logvar, z, set_repr

        return mu, logvar

In [ ]:
class SlotSetEncoder(nn.Module):
    """
    Baseline 3:
      Deep Sets encoder:
      slots -> shared phi -> permutation-invariant aggregation -> rho -> z
    """
    def __init__(self, slot_dim=64, z_dim=64, hidden_dim=128, dropout=0.0, agg="mean"):
        super().__init__()
        assert agg in ["mean", "sum"], "agg must be 'mean' or 'sum'"
        self.agg = agg

        self.phi = nn.Sequential(
            nn.Linear(slot_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.rho = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, z_dim),
        )

    def forward(self, slots):
        h = self.phi(slots)                  # (B, K, H)

        if self.agg == "mean":
            set_repr = h.mean(dim=1)
        else:
            set_repr = h.sum(dim=1)

        z = self.rho(set_repr)
        return z, set_repr

In [ ]:
class DeepSetsProbFactorPredictor(nn.Module):
    """
    Baseline 4:
      Deep Sets predictor.
    """
    def __init__(self, slot_dim=64, z_dim=64, hidden_dim=128, out_dim=10, dropout=0.0, agg="mean"):
        super().__init__()
        self.out_dim = out_dim
        self.z_dim = z_dim

        self.set_encoder = SlotSetEncoder(
            slot_dim=slot_dim,
            z_dim=z_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
            agg=agg
        )

        self.head = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim * 2)
        )

    def forward(self, slots, return_latent=False):
        z, set_repr = self.set_encoder(slots)
        out = self.head(z)

        mu = out[:, :self.out_dim]
        logvar = out[:, self.out_dim:]

        if return_latent:
            return mu, logvar, z, set_repr

        return mu, logvar

In [ ]:
class FactorQueryAttentionPredictor(nn.Module):
    """
    Baseline 5:
      factor-specific query attention over slots.

    Each latent factor gets its own learnable query.
    The query attends to the K slots and produces factor-specific context. This helps test whether different factors rely on different slots.
    """
    def __init__(
        self,
        slot_dim=64,
        z_dim=64,
        hidden_dim=128,
        out_dim=10,
        dropout=0.0,
        num_heads_unused=1,
    ):
        super().__init__()
        self.out_dim = out_dim
        self.z_dim = z_dim
        self.hidden_dim = hidden_dim

        self.slot_proj = nn.Sequential(
            nn.Linear(slot_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.factor_queries = nn.Parameter(torch.randn(out_dim, hidden_dim) * 0.02)
        self.context_norm = nn.LayerNorm(hidden_dim)

        self.factor_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

        self.z_proj = nn.Sequential(
            nn.Linear(out_dim * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, z_dim),
        )

    def compute_attention(self, slots):
        """
        Returns:
          h:    (B, K, H)
          attn: (B, out_dim, K)
        """
        h = self.slot_proj(slots)                         # (B, K, H)

        q = self.factor_queries                           # (F, H)
        scores = torch.einsum("fh,bkh->bfk", q, h) / math.sqrt(self.hidden_dim)
        attn = torch.softmax(scores, dim=-1)              # (B, F, K)

        return h, attn

    def forward(self, slots, return_latent=False):
        h, attn = self.compute_attention(slots)

        context = torch.einsum("bfk,bkh->bfh", attn, h)   # (B, F, H)
        context = self.context_norm(context)

        factor_out = self.factor_head(context)            # (B, F, 2)

        mu = factor_out[:, :, 0]                          # (B, F)
        logvar = factor_out[:, :, 1]                      # (B, F)

        flat_context = context.reshape(context.shape[0], -1)
        z = self.z_proj(flat_context)

        if return_latent:
            set_repr = flat_context
            return mu, logvar, z, set_repr

        return mu, logvar


In [ ]:
# Keep old class name as alias so older cells do not break.
ProbSetSlotFactorPredictor = DeepSetsProbFactorPredictor

In [ ]:
def build_predictor(
    architecture,
    slot_dim=64,
    num_slots=4,
    z_dim=64,
    hidden_dim=128,
    out_dim=10,
    dropout=0.0,
):

    # constructor for architecture comparison.
    architecture = architecture.lower()

    if architecture == "meanpool_linear":
        return MeanPoolLinearProbFactorPredictor(
            slot_dim=slot_dim,
            out_dim=out_dim,
        )

    if architecture == "meanpool":
        return MeanPoolProbFactorPredictor(
            slot_dim=slot_dim,
            z_dim=z_dim,
            hidden_dim=hidden_dim,
            out_dim=out_dim,
            dropout=dropout,
        )

    if architecture == "flatmlp":
        return FlatMLPProbFactorPredictor(
            num_slots=num_slots,
            slot_dim=slot_dim,
            z_dim=z_dim,
            hidden_dim=max(256, hidden_dim),
            out_dim=out_dim,
            dropout=dropout,
        )

    if architecture == "deepsets":
        return DeepSetsProbFactorPredictor(
            slot_dim=slot_dim,
            z_dim=z_dim,
            hidden_dim=hidden_dim,
            out_dim=out_dim,
            dropout=dropout,
            agg="mean",
        )

    if architecture == "factorquery":
        return FactorQueryAttentionPredictor(
            slot_dim=slot_dim,
            z_dim=z_dim,
            hidden_dim=hidden_dim,
            out_dim=out_dim,
            dropout=dropout,
        )

    raise ValueError(f"Unknown architecture: {architecture}")

In [ ]:
@torch.no_grad()
def collect_factor_query_attention(predictor, loader):
    """
    Collect average factor-to-slot attention.

    Only works for FactorQueryAttentionPredictor.
    Returns:
      mean_attn: (out_dim, num_slots)
    """
    if not hasattr(predictor, "compute_attention"):
        return None

    predictor.eval()
    attn_all = []

    for slots, y in tqdm(loader, desc="collect factor-query attention"):
        slots = slots.to(device, non_blocking=True)
        _, attn = predictor.compute_attention(slots)
        attn_all.append(attn.detach().cpu().numpy())

    attn_all = np.concatenate(attn_all, axis=0)   # (N, F, K)
    mean_attn = attn_all.mean(axis=0)             # (F, K)

    return mean_attn

## **LOSSES (baseline NLL + residual-variance regularization)**

In [ ]:
def gaussian_nll(mu, logvar, y):
    var = torch.exp(logvar) + 1e-8
    nll = 0.5 * (logvar + (y - mu) ** 2 / var)
    return nll.mean()


def residual_variance_regularized_loss(
    mu,
    logvar,
    y,
    z=None,
    lambda_var_reg=1.0,
    lambda_l1=1e-4,
):
    """
    In-training loss with residual-variance regularization.
    """

    var = torch.exp(logvar) + 1e-8

    nll = 0.5 * (logvar + (y - mu) ** 2 / var)
    nll = nll.mean()

    residual2 = (y - mu) ** 2
    residual_variance_regularization = torch.abs(residual2 - var).mean()

    l1 = 0.0
    if z is not None:
        l1 = z.abs().mean()

    return nll + lambda_var_reg * residual_variance_regularization + lambda_l1 * l1

## **Train/Val loops**

In [ ]:
def run_epoch(predictor, optimizer, loader, mode, loss_mode="nll", lambdas=None):
    assert mode in ["train", "eval"]
    predictor.train() if mode == "train" else predictor.eval()
    total, n = 0.0, 0

    for slots, y in tqdm(loader, desc=f"{mode}:{loss_mode}"):
        slots = slots.to(device, non_blocking=True)   # (B, K, D)
        y = y.to(device, non_blocking=True)

        if mode == "train":
            optimizer.zero_grad()

        if loss_mode == "nll":
            mu, logvar = predictor(slots)
            loss = gaussian_nll(mu, logvar, y)

        elif loss_mode == "residual_variance_reg":
            mu, logvar, z, set_repr = predictor(slots, return_latent=True)
            loss = residual_variance_regularized_loss(
                mu=mu,
                logvar=logvar,
                y=y,
                z=z,
                lambda_var_reg=lambdas["lambda_var_reg"],
                lambda_l1=lambdas["lambda_l1"],
            )
        else:
            raise ValueError(loss_mode)

        if mode == "train":
            loss.backward()
            torch.nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
            optimizer.step()

        total += float(loss.item()) * slots.size(0)
        n += slots.size(0)

    return total / max(1, n)

# **METRICS: PER-FACTOR & GLOBAL METRICS**

In [ ]:
@torch.no_grad()
def collect_z_y(predictor, loader):
    """
    loader yields (slots, y). We collect z and y.
    """
    predictor.eval()
    Zs, Ys = [], []

    for slots, y in tqdm(loader, desc="collect (z, y)"):
        slots = slots.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        _, _, z, set_repr = predictor(slots, return_latent=True)
        Zs.append(z.detach().cpu().numpy())
        Ys.append(y.detach().cpu().numpy())

    Z = np.concatenate(Zs, axis=0)
    Y = np.concatenate(Ys, axis=0)
    return Z, Y


@torch.no_grad()
def collect_setrepr_y(predictor, loader):
    predictor.eval()
    Rs, Ys = [], []

    for slots, y in tqdm(loader, desc="collect (set_repr, y)"):
        slots = slots.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        _, _, z, set_repr = predictor(slots, return_latent=True)
        Rs.append(set_repr.detach().cpu().numpy())
        Ys.append(y.detach().cpu().numpy())

    R = np.concatenate(Rs, axis=0)
    Y = np.concatenate(Ys, axis=0)
    return R, Y


In [ ]:
@torch.no_grad()
def collect_mu_var(predictor, loader, T=None):
    predictor.eval()
    mus, vars_, ys = [], [], []

    for slots, y in tqdm(loader, desc="collect (mu, var)"):
        slots = slots.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        mu, logvar = predictor(slots)
        var = torch.exp(logvar) + 1e-8

        if T is not None:
            if np.isscalar(T):
                var = var * (float(T) ** 2)
            else:
                Tt = torch.tensor(T, dtype=var.dtype, device=var.device).view(1, -1)
                var = var * (Tt ** 2)

        mus.append(mu.detach().cpu())
        vars_.append(var.detach().cpu())
        ys.append(y.detach().cpu())

    mu = torch.cat(mus, 0).numpy()
    var = torch.cat(vars_, 0).numpy()
    y  = torch.cat(ys, 0).numpy()
    return mu, var, y

In [ ]:
def rmse_mae(mu, y):
    rmse = np.sqrt(np.mean((mu - y) ** 2, axis=0))
    mae  = np.mean(np.abs(mu - y), axis=0)
    return rmse, mae


def gaussian_nll_np(mu, var, y):
    return float(np.mean(0.5 * (np.log(var) + ((y - mu) ** 2) / var)))


def sharpness_np(var):
    return float(np.mean(var))


def reg_ece_sigma_mae(mu, var, y, num_bins=15, per_factor=False):
    """
    Consistent Reg-ECE (Gaussian regression ECE):

    - per-factor: ECE needs to be computed separately for each factor using that factor's sigma-quantile bins.
    - global: mean of the per-factor ECE values (so global is consistent with per-factor).

    ECE compares:
      MAE_bin  vs  E[|N(0, sigma_bin)|] = sqrt(2/pi) * sigma_bin
    """
    sigma = np.sqrt(var + 1e-8)
    err   = np.abs(mu - y)
    K = y.shape[1]

    def ece_1d(s, e):
        bins = np.quantile(s, np.linspace(0, 1, num_bins + 1))
        ece = 0.0
        n = len(s)
        for i in range(num_bins):
            lo, hi = bins[i], bins[i + 1]
            mask = (s >= lo) & (s < hi) if i < num_bins - 1 else (s >= lo) & (s <= hi)
            m = mask.sum()
            if m == 0:
                continue
            sigma_bin = s[mask].mean()
            mae_bin   = e[mask].mean()
            mae_ideal = np.sqrt(2 / np.pi) * sigma_bin
            ece += (m / n) * abs(mae_bin - mae_ideal)
        return float(ece)

    per = np.zeros(K, dtype=np.float32)
    for k in range(K):
        per[k] = ece_1d(sigma[:, k], err[:, k])

    if per_factor:
        return per
    return float(per.mean())


# **POSTHOC CALIBRATION: TEMPERATURE SCALING ON THE VALIDATION SET**

In [ ]:
def optimize_temperature(mu, var, y, lr=0.01, max_iter=200):
    mu_t  = torch.tensor(mu,  dtype=torch.float32, device=device)
    var_t = torch.tensor(var, dtype=torch.float32, device=device)
    y_t   = torch.tensor(y,   dtype=torch.float32, device=device)

    # Unconstrained parameter (we optimize s, not T directly)
    s = torch.tensor(0.0, requires_grad=True, device=device)
    opt = torch.optim.Adam([s], lr=lr)

    for _ in range(max_iter):
        opt.zero_grad()

        # Positive temperature
        T = F.softplus(s) + 1e-6

        # Scale predictive variance by T^2 for Gaussian temperature scaling.
        varT = var_t * (T**2)

        loss = 0.5 * (torch.log(varT) + (y_t - mu_t)**2 / varT)
        loss = loss.mean()

        loss.backward()
        opt.step()

    # Return the learned T (not s)
    with torch.no_grad():
        T_final = F.softplus(s) + 1e-6
    return float(T_final.detach().cpu().item())


In [ ]:
def optimize_temperature_per_factor(mu, var, y, lr=0.01, max_iter=300):
    mu_t  = torch.tensor(mu,  dtype=torch.float32, device=device)
    var_t = torch.tensor(var, dtype=torch.float32, device=device)
    y_t   = torch.tensor(y,   dtype=torch.float32, device=device)

    K = mu_t.shape[1]

    s = torch.zeros(K, requires_grad=True, device=device)
    opt = torch.optim.Adam([s], lr=lr)

    for _ in range(max_iter):
        opt.zero_grad()

        T = F.softplus(s) + 1e-6
        varT = var_t * (T.view(1, -1) ** 2)

        loss = 0.5 * (torch.log(varT) + (y_t - mu_t) ** 2 / varT)
        loss = loss.mean()

        loss.backward()
        opt.step()

    with torch.no_grad():
        T_final = F.softplus(s) + 1e-6

    return T_final.detach().cpu().numpy()


# **Conformal baseline**

In [ ]:
def conformal_q_abs_error(mu_cal, y_cal, alpha=0.1):
    # per-factor q of |error|
    abs_err = np.abs(y_cal - mu_cal)
    q = np.quantile(abs_err, 1 - alpha, axis=0)
    return q

def conformal_coverage(mu_test, y_test, q):
    lo = mu_test - q
    hi = mu_test + q
    return float(((y_test >= lo) & (y_test <= hi)).mean())


#  **CALIBRATION (Isotonic on PIT) + GAUSSIAN SPLIT CONFORMAL**

In [ ]:
def _safe_clip01(u, eps=1e-6):
    return np.clip(u, eps, 1 - eps)

def pit_gaussian(mu, var, y):
    sigma = np.sqrt(var + 1e-8)
    z = (y - mu) / sigma
    u = norm.cdf(z)
    return _safe_clip01(u)

In [ ]:
class IsotonicCDFCalibrator:
    def __init__(self, calibrators, u_grid, g_grid):
        self.calibrators = calibrators
        self.u_grid = u_grid
        self.g_grid = g_grid

    def transform(self, u):
        u = _safe_clip01(u)
        K = u.shape[1]
        out = np.zeros_like(u, dtype=np.float64)

        for k in range(K):
            out[:, k] = self.calibrators[k].transform(u[:, k])

        return _safe_clip01(out)

    def inverse(self, v):
        """
        Invert calibrated CDF values back to raw Gaussian PIT values.

        Isotonic regression can produce flat regions, so g_grid may contain
        repeated values. np.interp expects a strictly increasing x-axis.
        We therefore remove repeated calibrated CDF values before interpolation.
        """
        v = _safe_clip01(v)
        K = v.shape[1]
        out = np.zeros_like(v, dtype=np.float64)

        for k in range(K):
            gk = np.asarray(self.g_grid[k], dtype=np.float64)
            uk = np.asarray(self.u_grid, dtype=np.float64)

            g_unique, idx = np.unique(gk, return_index=True)
            u_unique = uk[idx]

            # fallback if isotonic output collapses to a constant.
            if len(g_unique) < 2:
                out[:, k] = v[:, k]
            else:
                out[:, k] = np.interp(v[:, k], g_unique, u_unique)

        return _safe_clip01(out)

In [ ]:

def fit_isotonic_cdf_calibrator(mu_cal, var_cal, y_cal, grid_size=2001):
    u = pit_gaussian(mu_cal, var_cal, y_cal)
    N, K = u.shape

    calibrators = []
    for k in range(K):
        ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        order = np.argsort(u[:, k])
        x = u[order, k]
        y = (np.arange(N) + 1) / N
        ir.fit(x, y)
        calibrators.append(ir)

    u_grid = np.linspace(0.0, 1.0, grid_size)
    g_grid = np.zeros((K, grid_size), dtype=np.float64)
    for k in range(K):
        g_grid[k] = calibrators[k].transform(u_grid)

    return IsotonicCDFCalibrator(calibrators, u_grid, g_grid)

def calibrated_interval_from_cdf(mu, var, cdf_calibrator, level=0.9):
    sigma = np.sqrt(var + 1e-8)
    u_lo = (1 - level) / 2
    u_hi = (1 + level) / 2

    N, K = mu.shape
    v_lo = np.full((N, K), u_lo, dtype=np.float64)
    v_hi = np.full((N, K), u_hi, dtype=np.float64)

    u_lo_inv = cdf_calibrator.inverse(v_lo)
    u_hi_inv = cdf_calibrator.inverse(v_hi)

    z_lo = norm.ppf(_safe_clip01(u_lo_inv))
    z_hi = norm.ppf(_safe_clip01(u_hi_inv))

    lo = mu + sigma * z_lo
    hi = mu + sigma * z_hi
    return lo, hi

def coverage_from_interval(lo, hi, y):
    return ((y >= lo) & (y <= hi)).mean(axis=0)

In [ ]:
def gaussian_split_conformal_interval(mu_cal, var_cal, y_cal, mu_te, var_te, alpha=0.1):
    """
    Gaussian split conformal interval.

    conformity score:
      score = max(lower_cal - y_cal, y_cal - upper_cal, 0)
    Conformal scores should not be negative; hence the zero term
    """
    z = norm.ppf(1 - alpha / 2)

    sig_cal = np.sqrt(var_cal + 1e-8)
    sig_te  = np.sqrt(var_te + 1e-8)

    qlo_cal = mu_cal - z * sig_cal
    qhi_cal = mu_cal + z * sig_cal

    scores = np.maximum.reduce([
        qlo_cal - y_cal,
        y_cal - qhi_cal,
        np.zeros_like(y_cal)
    ])

    qhat = np.quantile(scores, 1 - alpha, axis=0)

    qlo_te = mu_te - z * sig_te
    qhi_te = mu_te + z * sig_te

    lo = qlo_te - qhat.reshape(1, -1)
    hi = qhi_te + qhat.reshape(1, -1)

    return lo, hi, qhat

# **MATCHED PAIR CONSTRUCTION**

In [ ]:
PAIR_CONFIG = {
    "min_target_gap_std": 0.50,
    "max_protected_drift_std": 0.25,
    "joint_protected_dist_std": 0.60,

    "candidate_knn_k": 1024,
    "max_candidates": 5000,

    "min_mask_iou_mean": 0.55,
    "min_mask_iou_min": 0.35,
    "max_slot_l2": 1.50,
    "max_image_mse": 0.080,

    "num_pairs": NUM_MATCHED_PAIRS,
    "allow_reuse": False,
}

PROTECTED_FACTORS = {
    0: [1, 2, 3, 4, 5, 6, 7, 8, 9],
    1: [0, 2, 3, 4, 5, 6, 7, 8, 9],
    2: [0, 1, 3, 4, 5, 6, 7, 8, 9],
    3: [0, 1, 2, 4, 5, 6, 7, 8, 9],
    4: [0, 1, 2, 3, 5, 6, 7, 8, 9],
    5: [0, 1, 2, 3, 4, 6, 7, 8, 9],
    6: [0, 1, 2, 3, 4, 5, 7, 8, 9],
    7: [0, 1, 2, 3, 4, 5, 6, 8, 9],
    8: [0, 1, 2, 3, 4, 5, 6, 7, 9],
    9: [0, 1, 2, 3, 4, 5, 6, 7, 8],
}


def standardized_latents(latents_np):
    Y = np.asarray(latents_np, dtype=np.float64)
    mean = Y.mean(axis=0, keepdims=True)
    std = Y.std(axis=0, keepdims=True) + 1e-12
    return (Y - mean) / std, mean.squeeze(0), std.squeeze(0)

In [ ]:
def build_candidate_pairs_global(
    latents_np,
    factor_j,
    protected_factors,
    seed=0,
    min_target_gap_std=0.35,
    max_protected_drift_std=0.20,
    joint_protected_dist_std=0.45,
    candidate_knn_k=512,
    max_candidates=5000,
):
    """
    Build latent-space candidate pairs and record stage-wise filtering counts.

    N.B:
      n_raw_candidates_considered is not all possible pairs in the dataset; It is the number of candidate pairs examined under the capped kNN search.
    """
    rng = np.random.default_rng(seed)

    Y = np.asarray(latents_np, dtype=np.float64)
    Y_std, _, _ = standardized_latents(Y)
    N, K = Y_std.shape

    protected_factors = list(protected_factors)

    X_protected = Y_std[:, protected_factors]
    y_target = Y_std[:, factor_j]

    k_eff = min(max(8, candidate_knn_k), N)

    nn = NearestNeighbors(
        n_neighbors=k_eff,
        metric="euclidean",
        algorithm="auto"
    )
    nn.fit(X_protected)

    candidates = []
    seen_pairs = set()

    diagnostic_counts = {
        "n_raw_candidates_considered": 0,
        "n_pass_target_gap": 0,
        "n_pass_protected_max_drift": 0,
        "n_pass_joint_protected_dist": 0,
        "candidate_search_note": (
            "Raw candidates are candidates examined under capped kNN search, "
            "not all possible dataset pairs."
        ),
    }

    anchor_order = rng.permutation(N)

    for a in anchor_order:
        if len(candidates) >= max_candidates:
            break

        _, neighbors = nn.kneighbors(
            X_protected[a:a + 1],
            n_neighbors=k_eff,
            return_distance=True
        )

        neighbors = neighbors[0]

        for b in neighbors:
            b = int(b)

            if b == a:
                continue

            pair_key = tuple(sorted((int(a), int(b))))
            if pair_key in seen_pairs:
                continue

            seen_pairs.add(pair_key)
            diagnostic_counts["n_raw_candidates_considered"] += 1

            target_gap = abs(y_target[a] - y_target[b])
            if target_gap < min_target_gap_std:
                continue

            diagnostic_counts["n_pass_target_gap"] += 1

            protected_delta = np.abs(
                Y_std[a, protected_factors] - Y_std[b, protected_factors]
            )

            protected_max_drift = float(protected_delta.max())
            protected_joint_dist = float(np.linalg.norm(protected_delta, ord=2))

            if protected_max_drift > max_protected_drift_std:
                continue

            diagnostic_counts["n_pass_protected_max_drift"] += 1

            if protected_joint_dist > joint_protected_dist_std:
                continue

            diagnostic_counts["n_pass_joint_protected_dist"] += 1

            latent_score = (
                1.00 * protected_joint_dist
                + 0.75 * protected_max_drift
                - 0.20 * target_gap
            )

            candidates.append({
                "a": int(a),
                "b": int(b),
                "factor_j": int(factor_j),
                "target_gap_std": float(target_gap),
                "protected_joint_dist_std": float(protected_joint_dist),
                "protected_max_drift_std": float(protected_max_drift),
                "latent_score": float(latent_score),
            })

            if len(candidates) >= max_candidates:
                break

    candidates = sorted(candidates, key=lambda r: r["latent_score"])
    return candidates, diagnostic_counts

In [ ]:
@torch.no_grad()
def extract_screen_tensors_for_indices(sa_model, dataset, indices, batch_size=256, slot_init_seed=1234):
    sa_model.eval()

    unique_indices = sorted(set([int(i) for i in indices]))
    index_to_pos = {idx: p for p, idx in enumerate(unique_indices)}

    imgs_all = []
    slots_all = []
    masks_all = []

    for start in tqdm(range(0, len(unique_indices), batch_size), desc="extract visual/slot tensors"):
        batch_indices = unique_indices[start:start + batch_size]

        xs = []
        for idx in batch_indices:
            x, _ = dataset[idx]
            xs.append(x)

        x = torch.stack(xs, dim=0).to(device, non_blocking=True)

        feats = sa_model.project(sa_model.encoder(x))
        K = sa_model.slot_attention.num_slots

        init_noise = make_fixed_slot_init_noise(
            feats=feats,
            num_slots=K,
            seed=slot_init_seed,
        )

        slots = sa_model.slot_attention(feats, init_noise=init_noise)
        _, masks = sa_model.decoder(slots)

        imgs_all.append(x.detach().cpu())
        slots_all.append(slots.detach().cpu())
        masks_all.append(masks.detach().cpu())

    imgs_all = torch.cat(imgs_all, dim=0)
    slots_all = torch.cat(slots_all, dim=0)
    masks_all = torch.cat(masks_all, dim=0)

    return unique_indices, index_to_pos, imgs_all, slots_all, masks_all

In [ ]:
def soft_iou_np(mask_a, mask_b, eps=1e-8):
    inter = np.minimum(mask_a, mask_b).sum()
    union = np.maximum(mask_a, mask_b).sum()
    return float(inter / (union + eps))


In [ ]:
def mask_spatial_entropy_np(mask, eps=1e-8):

    ## Compute normalized spatial entropy for one slot mask.

    m = np.asarray(mask, dtype=np.float64)
    p = m / (m.sum() + eps)
    h = -np.sum(p * np.log(p + eps))
    h_norm = h / np.log(p.size + eps)
    return float(h_norm)

In [ ]:
def mask_pixel_slot_entropy_np(masks, eps=1e-8):

    # Compute average normalized entropy across slots at each pixel.

    m = np.asarray(masks, dtype=np.float64)

    if m.ndim == 4:
        m = m[:, 0]

    K = m.shape[0]
    h = -np.sum(m * np.log(m + eps), axis=0)
    h_norm = h / np.log(K + eps)

    return float(np.mean(h_norm))

In [ ]:
def mask_area_np(mask):
    """
    Compute mean soft area for one slot mask.
    """
    return float(np.asarray(mask, dtype=np.float64).mean())

In [ ]:
def within_image_mask_diversity_np(masks, eps=1e-8):
    """
    Compute diversity between slots inside one image using pairwise soft IoU.
    """
    m = np.asarray(masks, dtype=np.float64)

    if m.ndim == 4:
        m = m[:, 0]

    K = m.shape[0]
    pair_ious = []

    for i in range(K):
        for j in range(i + 1, K):
            pair_ious.append(soft_iou_np(m[i], m[j], eps=eps))

    if len(pair_ious) == 0:
        return np.nan, np.nan

    mean_pair_iou = float(np.mean(pair_ious))
    diversity = float(1.0 - mean_pair_iou)

    return diversity, mean_pair_iou

In [ ]:
def mask_audit_stats(masks):
    # Compute mask range, slot area, slot entropy, and within-image mask diversity.
    if torch.is_tensor(masks):
        masks = masks.detach().cpu().numpy()

    if masks.ndim == 4:
        masks_2d = masks[:, 0]
    else:
        masks_2d = masks

    K = masks_2d.shape[0]

    areas = []
    entropies = []

    for k in range(K):
        areas.append(mask_area_np(masks_2d[k]))
        entropies.append(mask_spatial_entropy_np(masks_2d[k]))

    diversity, mean_pair_iou = within_image_mask_diversity_np(masks_2d)

    areas = np.asarray(areas, dtype=np.float64)
    entropies = np.asarray(entropies, dtype=np.float64)

    return {
        "mask_raw_min": float(np.min(masks_2d)),
        "mask_raw_max": float(np.max(masks_2d)),

        "mask_area_per_slot": areas.tolist(),
        "mask_area_min": float(np.min(areas)),
        "mask_area_max": float(np.max(areas)),
        "mask_area_mean": float(np.mean(areas)),
        "mask_area_std": float(np.std(areas)),

        "mask_entropy_per_slot": entropies.tolist(),
        "mask_entropy_mean": float(np.mean(entropies)),
        "mask_entropy_min": float(np.min(entropies)),
        "mask_entropy_max": float(np.max(entropies)),
        "mask_entropy_std": float(np.std(entropies)),

        "pixel_slot_entropy": mask_pixel_slot_entropy_np(masks_2d),
        "within_image_mask_diversity": diversity,
        "within_image_mask_pair_iou_mean": mean_pair_iou,
    }

In [ ]:
def matched_slot_screen_stats(slots_a, slots_b, masks_a, masks_b):
    # Match slots by Hungarian matching over soft mask IoU.
    slots_a_np = slots_a.numpy()
    slots_b_np = slots_b.numpy()
    masks_a_np = masks_a.numpy()
    masks_b_np = masks_b.numpy()

    K = slots_a_np.shape[0]
    sim = np.zeros((K, K), dtype=np.float32)

    for i in range(K):
        for j in range(K):
            sim[i, j] = soft_iou_np(masks_a_np[i, 0], masks_b_np[j, 0])

    row_ind, col_ind = linear_sum_assignment(-sim)
    matched_ious = sim[row_ind, col_ind]

    slot_l2 = []
    slot_cos = []

    for i, j in zip(row_ind, col_ind):
        va = slots_a_np[i]
        vb = slots_b_np[j]

        slot_l2.append(float(np.linalg.norm(va - vb)))

        denom = (np.linalg.norm(va) * np.linalg.norm(vb)) + 1e-8
        slot_cos.append(float(np.dot(va, vb) / denom))

    audit_a = mask_audit_stats(masks_a_np)
    audit_b = mask_audit_stats(masks_b_np)

    return {
        "mask_iou_mean": float(np.mean(matched_ious)),
        "mask_iou_median": float(np.median(matched_ious)),
        "mask_iou_min": float(np.min(matched_ious)),

        "slot_l2_mean": float(np.mean(slot_l2)),
        "slot_l2_median": float(np.median(slot_l2)),
        "slot_l2_max": float(np.max(slot_l2)),
        "slot_cos_mean": float(np.mean(slot_cos)),

        "mask_a_raw_min": audit_a["mask_raw_min"],
        "mask_a_raw_max": audit_a["mask_raw_max"],
        "mask_a_area_per_slot": audit_a["mask_area_per_slot"],
        "mask_a_area_min": audit_a["mask_area_min"],
        "mask_a_area_max": audit_a["mask_area_max"],
        "mask_a_area_mean": audit_a["mask_area_mean"],
        "mask_a_area_std": audit_a["mask_area_std"],
        "mask_a_entropy_per_slot": audit_a["mask_entropy_per_slot"],
        "mask_a_entropy_mean": audit_a["mask_entropy_mean"],
        "mask_a_entropy_min": audit_a["mask_entropy_min"],
        "mask_a_entropy_max": audit_a["mask_entropy_max"],
        "mask_a_entropy_std": audit_a["mask_entropy_std"],
        "mask_a_pixel_slot_entropy": audit_a["pixel_slot_entropy"],
        "mask_a_within_image_diversity": audit_a["within_image_mask_diversity"],
        "mask_a_within_image_pair_iou_mean": audit_a["within_image_mask_pair_iou_mean"],

        "mask_b_raw_min": audit_b["mask_raw_min"],
        "mask_b_raw_max": audit_b["mask_raw_max"],
        "mask_b_area_per_slot": audit_b["mask_area_per_slot"],
        "mask_b_area_min": audit_b["mask_area_min"],
        "mask_b_area_max": audit_b["mask_area_max"],
        "mask_b_area_mean": audit_b["mask_area_mean"],
        "mask_b_area_std": audit_b["mask_area_std"],
        "mask_b_entropy_per_slot": audit_b["mask_entropy_per_slot"],
        "mask_b_entropy_mean": audit_b["mask_entropy_mean"],
        "mask_b_entropy_min": audit_b["mask_entropy_min"],
        "mask_b_entropy_max": audit_b["mask_entropy_max"],
        "mask_b_entropy_std": audit_b["mask_entropy_std"],
        "mask_b_pixel_slot_entropy": audit_b["pixel_slot_entropy"],
        "mask_b_within_image_diversity": audit_b["within_image_mask_diversity"],
        "mask_b_within_image_pair_iou_mean": audit_b["within_image_mask_pair_iou_mean"],

        "match_rows": row_ind.tolist(),
        "match_cols": col_ind.tolist(),
    }

In [ ]:
def screen_candidate_pairs_visual_slot(
    candidates,
    sa_model,
    dataset,
    min_mask_iou_mean=0.55,
    min_mask_iou_min=0.35,
    max_slot_l2=1.50,
    max_image_mse=0.080,
    batch_size=256,
):

    # Screen candidate pairs using image, slot, and mask consistency.

    if len(candidates) == 0:
        return [], []

    all_indices = []
    for r in candidates:
        all_indices.append(r["a"])
        all_indices.append(r["b"])

    _, index_to_pos, imgs_all, slots_all, masks_all = extract_screen_tensors_for_indices(
        sa_model=sa_model,
        dataset=dataset,
        indices=all_indices,
        batch_size=batch_size,
    )

    accepted = []
    rejected = []

    for r in tqdm(candidates, desc="screen candidates"):
        a = int(r["a"])
        b = int(r["b"])

        pa = index_to_pos[a]
        pb = index_to_pos[b]

        img_a = imgs_all[pa]
        img_b = imgs_all[pb]

        slots_a = slots_all[pa]
        slots_b = slots_all[pb]

        masks_a = masks_all[pa]
        masks_b = masks_all[pb]

        image_mse = float(torch.mean((img_a - img_b) ** 2).item())

        stats = matched_slot_screen_stats(
            slots_a=slots_a,
            slots_b=slots_b,
            masks_a=masks_a,
            masks_b=masks_b,
        )

        row = {
            **r,
            "image_mse": image_mse,
            **stats,
        }

        fail_reasons = []

        if row["mask_iou_mean"] < min_mask_iou_mean:
            fail_reasons.append("mask_iou_mean")
        if row["mask_iou_min"] < min_mask_iou_min:
            fail_reasons.append("mask_iou_min")
        if row["slot_l2_mean"] > max_slot_l2:
            fail_reasons.append("slot_l2")
        if row["image_mse"] > max_image_mse:
            fail_reasons.append("image_mse")

        row["fail_mask_iou_mean"] = "mask_iou_mean" in fail_reasons
        row["fail_mask_iou_min"] = "mask_iou_min" in fail_reasons
        row["fail_slot_l2"] = "slot_l2" in fail_reasons
        row["fail_image_mse"] = "image_mse" in fail_reasons
        row["fail_reasons"] = ",".join(fail_reasons)

        if len(fail_reasons) == 0:
            accepted.append(row)
        else:
            rejected.append(row)

    for row in accepted:
        row["final_score"] = (
            1.00 * row["protected_joint_dist_std"]
            + 0.75 * row["protected_max_drift_std"]
            + 0.50 * row["image_mse"]
            + 0.50 * row["slot_l2_mean"]
            - 0.25 * row["target_gap_std"]
            - 0.50 * row["mask_iou_mean"]
        )

    accepted = sorted(accepted, key=lambda r: r["final_score"])
    rejected = sorted(rejected, key=lambda r: r.get("latent_score", 999.0))

    return accepted, rejected


In [ ]:
def select_non_reused_pairs(accepted_rows, num_pairs=100, allow_reuse=False):
    pairs = []
    selected_rows = []
    used = set()

    for r in accepted_rows:
        a = int(r["a"])
        b = int(r["b"])

        if not allow_reuse:
            if a in used or b in used:
                continue

        pairs.append((a, b))
        selected_rows.append(r)

        if not allow_reuse:
            used.add(a)
            used.add(b)

        if len(pairs) >= num_pairs:
            break

    return pairs, selected_rows

In [ ]:
def summarize_selected_pair_quality(selected_rows):
    # Summarize selected matched-pair quality.
    if selected_rows is None or len(selected_rows) == 0:
        return {
            "n_pairs": 0,

            "target_gap_std_mean": np.nan,
            "target_gap_std_median": np.nan,
            "target_gap_std_min": np.nan,

            "protected_joint_dist_std_mean": np.nan,
            "protected_joint_dist_std_median": np.nan,

            "protected_max_drift_std_mean": np.nan,
            "protected_max_drift_std_median": np.nan,
            "protected_max_drift_std_max": np.nan,

            "mask_iou_mean": np.nan,
            "mask_iou_median": np.nan,
            "mask_iou_min_mean": np.nan,

            "slot_l2_mean": np.nan,
            "slot_l2_median": np.nan,
            "slot_l2_max": np.nan,
            "slot_cos_mean": np.nan,

            "image_mse_mean": np.nan,
            "image_mse_median": np.nan,

            "mask_entropy_mean": np.nan,
            "mask_entropy_min_mean": np.nan,
            "mask_entropy_max_mean": np.nan,
            "mask_entropy_std_mean": np.nan,

            "mask_area_mean": np.nan,
            "mask_area_min_mean": np.nan,
            "mask_area_max_mean": np.nan,
            "mask_area_std_mean": np.nan,

            "mask_within_image_diversity_mean": np.nan,
        }

    def vals(key):
        return np.asarray([r[key] for r in selected_rows], dtype=np.float64)

    target_gap = vals("target_gap_std")
    joint_dist = vals("protected_joint_dist_std")
    max_drift = vals("protected_max_drift_std")
    mask_iou = vals("mask_iou_mean")
    mask_iou_min = vals("mask_iou_min")
    slot_l2 = vals("slot_l2_mean")
    slot_l2_max = vals("slot_l2_max")
    slot_cos = vals("slot_cos_mean")
    image_mse = vals("image_mse")

    mask_entropy = np.asarray([
        0.5 * (r["mask_a_entropy_mean"] + r["mask_b_entropy_mean"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_entropy_min = np.asarray([
        0.5 * (r["mask_a_entropy_min"] + r["mask_b_entropy_min"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_entropy_max = np.asarray([
        0.5 * (r["mask_a_entropy_max"] + r["mask_b_entropy_max"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_entropy_std = np.asarray([
        0.5 * (r["mask_a_entropy_std"] + r["mask_b_entropy_std"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_area = np.asarray([
        0.5 * (r["mask_a_area_mean"] + r["mask_b_area_mean"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_area_min = np.asarray([
        0.5 * (r["mask_a_area_min"] + r["mask_b_area_min"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_area_max = np.asarray([
        0.5 * (r["mask_a_area_max"] + r["mask_b_area_max"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_area_std = np.asarray([
        0.5 * (r["mask_a_area_std"] + r["mask_b_area_std"])
        for r in selected_rows
    ], dtype=np.float64)

    mask_diversity = np.asarray([
        0.5 * (r["mask_a_within_image_diversity"] + r["mask_b_within_image_diversity"])
        for r in selected_rows
    ], dtype=np.float64)

    return {
        "n_pairs": int(len(selected_rows)),

        "target_gap_std_mean": float(np.mean(target_gap)),
        "target_gap_std_median": float(np.median(target_gap)),
        "target_gap_std_min": float(np.min(target_gap)),

        "protected_joint_dist_std_mean": float(np.mean(joint_dist)),
        "protected_joint_dist_std_median": float(np.median(joint_dist)),

        "protected_max_drift_std_mean": float(np.mean(max_drift)),
        "protected_max_drift_std_median": float(np.median(max_drift)),
        "protected_max_drift_std_max": float(np.max(max_drift)),

        "mask_iou_mean": float(np.mean(mask_iou)),
        "mask_iou_median": float(np.median(mask_iou)),
        "mask_iou_min_mean": float(np.mean(mask_iou_min)),

        "slot_l2_mean": float(np.mean(slot_l2)),
        "slot_l2_median": float(np.median(slot_l2)),
        "slot_l2_max": float(np.max(slot_l2_max)),
        "slot_cos_mean": float(np.mean(slot_cos)),

        "image_mse_mean": float(np.mean(image_mse)),
        "image_mse_median": float(np.median(image_mse)),

        "mask_entropy_mean": float(np.mean(mask_entropy)),
        "mask_entropy_min_mean": float(np.mean(mask_entropy_min)),
        "mask_entropy_max_mean": float(np.mean(mask_entropy_max)),
        "mask_entropy_std_mean": float(np.mean(mask_entropy_std)),

        "mask_area_mean": float(np.mean(mask_area)),
        "mask_area_min_mean": float(np.mean(mask_area_min)),
        "mask_area_max_mean": float(np.mean(mask_area_max)),
        "mask_area_std_mean": float(np.mean(mask_area_std)),

        "mask_within_image_diversity_mean": float(np.mean(mask_diversity)),
    }

In [ ]:
def build_matched_pairs(
    latents_np,
    factor_j,
    sa_model,
    dataset,
    protected_factors,
    seed=0,
    config=None,
    verbose=True,
):

    # Build matched pairs and return pair-level diagnostics.
    # The pair search is approximate/capped: raw candidates = candidates examined under capped kNN search.
    if config is None:
        config = PAIR_CONFIG

    candidates, latent_filter_counts = build_candidate_pairs_global(
        latents_np=latents_np,
        factor_j=factor_j,
        protected_factors=protected_factors,
        seed=seed,
        min_target_gap_std=config["min_target_gap_std"],
        max_protected_drift_std=config["max_protected_drift_std"],
        joint_protected_dist_std=config["joint_protected_dist_std"],
        candidate_knn_k=config["candidate_knn_k"],
        max_candidates=config["max_candidates"],
    )

    accepted_rows, rejected_rows = screen_candidate_pairs_visual_slot(
        candidates=candidates,
        sa_model=sa_model,
        dataset=dataset,
        min_mask_iou_mean=config["min_mask_iou_mean"],
        min_mask_iou_min=config["min_mask_iou_min"],
        max_slot_l2=config["max_slot_l2"],
        max_image_mse=config["max_image_mse"],
        batch_size=256,
    )

    pairs, selected_rows = select_non_reused_pairs(
        accepted_rows=accepted_rows,
        num_pairs=config["num_pairs"],
        allow_reuse=config["allow_reuse"],
    )

    quality = summarize_selected_pair_quality(selected_rows)

    rejected_reason_counts = {}
    for r in rejected_rows:
        reasons = str(r.get("fail_reasons", "")).split(",")

        for reason in reasons:
            if reason == "":
                continue

            key = f"n_reject_{reason}"
            rejected_reason_counts[key] = rejected_reason_counts.get(key, 0) + 1

    diagnostics = {
        "factor_j": int(factor_j),
        "factor_name": FACTOR_NAMES[factor_j],
        "protected_factors": protected_factors,

        **latent_filter_counts,

        "n_passing_image_slot_mask_checks": len(accepted_rows),
        "n_finally_selected": len(pairs),

        "threshold_min_target_gap_std": config["min_target_gap_std"],
        "threshold_max_protected_drift_std": config["max_protected_drift_std"],
        "threshold_joint_protected_dist_std": config["joint_protected_dist_std"],
        "threshold_candidate_knn_k": config["candidate_knn_k"],
        "threshold_max_candidates": config["max_candidates"],
        "threshold_min_mask_iou_mean": config["min_mask_iou_mean"],
        "threshold_min_mask_iou_min": config["min_mask_iou_min"],
        "threshold_max_slot_l2": config["max_slot_l2"],
        "threshold_max_image_mse": config["max_image_mse"],

        **quality,
        **rejected_reason_counts,
    }

    if verbose:
        print("\nMATCHED PAIR CONSTRUCTION DIAGNOSTICS")
        print("factor_j:", factor_j)
        print("protected_factors:", protected_factors)
        print("raw candidates considered:", diagnostics["n_raw_candidates_considered"])
        print("pass target gap:", diagnostics["n_pass_target_gap"])
        print("pass protected max drift:", diagnostics["n_pass_protected_max_drift"])
        print("pass joint protected distance:", diagnostics["n_pass_joint_protected_dist"])
        print("pass image/slot/mask checks:", diagnostics["n_passing_image_slot_mask_checks"])
        print("selected final pairs:", diagnostics["n_finally_selected"])
        print("note:", diagnostics["candidate_search_note"])
        print("diagnostics:", diagnostics)

    return pairs, selected_rows, accepted_rows, rejected_rows, diagnostics

In [ ]:
def plot_pair_examples_from_rows(
    dataset,
    rows,
    factor_j,
    title,
    n_pairs=6,
    save_path=None,
):
    if rows is None or len(rows) == 0:
        print(f"No rows to plot for {title}")
        return

    rows = rows[:min(n_pairs, len(rows))]

    fig, axes = plt.subplots(len(rows), 2, figsize=(7, 3.3 * len(rows)))

    if len(rows) == 1:
        axes = np.array([axes])

    for i, r in enumerate(rows):
        a = int(r["a"])
        b = int(r["b"])

        xa, ya = dataset[a]
        xb, yb = dataset[b]

        axes[i, 0].imshow(xa.permute(1, 2, 0))
        axes[i, 0].set_title(
            f"A idx={a} | f{factor_j}={ya[factor_j]:.3f}"
        )
        axes[i, 0].axis("off")

        axes[i, 1].imshow(xb.permute(1, 2, 0))
        axes[i, 1].set_title(
            f"B idx={b} | f{factor_j}={yb[factor_j]:.3f}\n"
            f"IoU={r.get('mask_iou_mean', np.nan):.2f}, "
            f"slotL2={r.get('slot_l2_mean', np.nan):.2f}, "
            f"MSE={r.get('image_mse', np.nan):.3f}"
        )
        axes[i, 1].axis("off")

    plt.suptitle(title, y=1.01)
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=220, bbox_inches="tight")
        print("Saved:", save_path)

    #plt.show()
    plt.close(fig)

In [ ]:
@torch.no_grad()
def plot_pair_mask_grids_from_rows(
    sa_model,
    dataset,
    rows,
    factor_j,
    title,
    n_pairs=4,
    save_path=None,
    slot_init_seed=1234,
):
    if rows is None or len(rows) == 0:
        print(f"No rows to plot for {title}")
        return

    sa_model.eval()
    rows = rows[:min(n_pairs, len(rows))]

    K = sa_model.slot_attention.num_slots
    n_cols = 2 + 2 * K

    fig, axes = plt.subplots(
        len(rows),
        n_cols,
        figsize=(2.2 * n_cols, 2.6 * len(rows))
    )

    if len(rows) == 1:
        axes = np.array([axes])

    for i, r in enumerate(rows):
        a = int(r["a"])
        b = int(r["b"])

        xa, ya = dataset[a]
        xb, yb = dataset[b]

        x = torch.stack([xa, xb], dim=0).to(device)

        feats = sa_model.project(sa_model.encoder(x))

        init_noise = make_fixed_slot_init_noise(
            feats=feats,
            num_slots=K,
            seed=slot_init_seed,
        )

        slots = sa_model.slot_attention(feats, init_noise=init_noise)
        _, masks = sa_model.decoder(slots)
        masks_np = masks.detach().cpu().numpy()

        axes[i, 0].imshow(xa.permute(1, 2, 0))
        axes[i, 0].set_title(f"A idx={a}\nf{factor_j}={ya[factor_j]:.3f}")
        axes[i, 0].axis("off")

        for k in range(K):
            mask = masks_np[0, k, 0]
            axes[i, 1 + k].imshow(mask, cmap="gray")
            axes[i, 1 + k].set_title(
                f"A mask {k}\nmin={mask.min():.3f}, max={mask.max():.3f}"
            )
            axes[i, 1 + k].axis("off")

        axes[i, 1 + K].imshow(xb.permute(1, 2, 0))
        axes[i, 1 + K].set_title(
            f"B idx={b}\nf{factor_j}={yb[factor_j]:.3f}\n"
            f"IoU={r.get('mask_iou_mean', np.nan):.2f}"
        )
        axes[i, 1 + K].axis("off")

        for k in range(K):
            mask = masks_np[1, k, 0]
            axes[i, 2 + K + k].imshow(mask, cmap="gray")
            axes[i, 2 + K + k].set_title(
                f"B mask {k}\nmin={mask.min():.3f}, max={mask.max():.3f}"
            )
            axes[i, 2 + K + k].axis("off")

    plt.suptitle(title, y=1.01)
    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=220, bbox_inches="tight")
        print("Saved:", save_path)

    #plt.show()
    plt.close(fig)

In [ ]:
@torch.no_grad()
def evaluate_predictor_matched_pair_stability(
    predictor,
    slots_cpu,
    latents_np,
    factor_j,
    pairs,
    pair_source="unknown",
    batch_size=256,
    selected_pair_rows=None,
    pair_diag=None,
):
    """
    Evaluate prediction stability on selected matched pairs.
    """
    predictor.eval()

    if len(pairs) == 0:
        output = {
            "inv_non_j_mu": np.nan,
            "sens_j_mu": np.nan,
            "inv_z": np.nan,
            "n_pairs_used": 0,
            "pair_source": pair_source,
            "metric_units": "factor-standardized for mu metrics; raw learned units for z",
        }

        if pair_diag is not None:
            output.update({f"pair_{k}": v for k, v in pair_diag.items()})

        return output

    idx_a = np.asarray([p[0] for p in pairs], dtype=np.int64)
    idx_b = np.asarray([p[1] for p in pairs], dtype=np.int64)

    def predict_mu_and_z(indices):
        mu_list, z_list = [], []

        for start in range(0, len(indices), batch_size):
            batch_idx = torch.as_tensor(
                indices[start:start + batch_size],
                dtype=torch.long
            )

            slots_batch = slots_cpu[batch_idx].to(device, non_blocking=True)
            mu, _, z, _ = predictor(slots_batch, return_latent=True)

            mu_list.append(mu.detach().cpu().numpy())
            z_list.append(z.detach().cpu().numpy())

        return np.concatenate(mu_list, axis=0), np.concatenate(z_list, axis=0)

    mu_a, z_a = predict_mu_and_z(idx_a)
    mu_b, z_b = predict_mu_and_z(idx_b)

    K = latents_np.shape[1]
    nonj_mask = np.ones(K, dtype=bool)
    nonj_mask[factor_j] = False

    std_y = np.asarray(latents_np, dtype=np.float64).std(axis=0) + 1e-12

    diff_nonj = np.abs(mu_a[:, nonj_mask] - mu_b[:, nonj_mask])
    inv_non_j_mu = float(
        np.mean(diff_nonj / std_y[nonj_mask].reshape(1, -1))
    )

    diff_j = np.abs(mu_a[:, factor_j] - mu_b[:, factor_j])
    sens_j_mu = float(
        np.mean(diff_j / std_y[factor_j])
    )

    inv_z = float(np.mean(np.abs(z_a - z_b)))

    output = {
        "inv_non_j_mu": inv_non_j_mu,
        "sens_j_mu": sens_j_mu,
        "inv_z": inv_z,
        "n_pairs_used": len(pairs),
        "pair_source": pair_source,
        "metric_units": "factor-standardized for mu metrics; raw learned units for z",
    }

    if pair_diag is not None:
        output.update({f"pair_{k}": v for k, v in pair_diag.items()})

    return output

# **R² Identifiability: train probe on TRAINSET, and test on TESTSET**

In [ ]:
def linear_probe_r2_train_test(Z_tr, Y_tr, Z_te, Y_te):

    # Train linear probe on TRAIN latents and evaluate on TEST latents.

    K = Y_tr.shape[1]
    r2 = np.zeros(K, dtype=np.float32)

    for k in range(K):
        reg = LinearRegression().fit(Z_tr, Y_tr[:, k])
        pred = reg.predict(Z_te)
        r2[k] = r2_score(Y_te[:, k], pred)

    return r2

# **SEED RUNNER**

**seed run (baseline NLL, posthoc, residual-variance regularization)**

In [ ]:
def make_loaders(seed: int):
    set_seed(seed)

    N = S_train_all.shape[0]
    val_size = int(VAL_FRAC * N)
    train_size = N - val_size

    g = torch.Generator()
    g.manual_seed(seed)
    perm = torch.randperm(N, generator=g)

    idx_train = perm[:train_size]
    idx_val   = perm[train_size:]

    val_sel_size = int(VAL_CAL_FRAC * val_size)
    idx_val_sel  = idx_val[:val_sel_size]
    idx_val_cal  = idx_val[val_sel_size:]

    S_tr = S_train_all[idx_train]
    Y_tr = Y_train_all[idx_train]

    S_val_sel = S_train_all[idx_val_sel]
    Y_val_sel = Y_train_all[idx_val_sel]

    S_val_cal = S_train_all[idx_val_cal]
    Y_val_cal = Y_train_all[idx_val_cal]

    S_te = S_test_all
    Y_te = Y_test_all

    train_data   = SlotSetDataset(S_tr, Y_tr)
    val_sel_data = SlotSetDataset(S_val_sel, Y_val_sel)
    val_cal_data = SlotSetDataset(S_val_cal, Y_val_cal)
    test_data    = SlotSetDataset(S_te, Y_te)

    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=g
    )

    val_sel_loader = DataLoader(
        val_sel_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_cal_loader = DataLoader(
        val_cal_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_sel_loader, val_cal_loader, test_loader, S_tr, Y_tr, S_te, Y_te


# **Train and Evaluate Functions**

In [ ]:
def save_checkpoint(path, predictor, optimizer, epoch, seed, loss_mode, best_val, extra=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    payload = {
        "epoch": epoch,
        "seed": seed,
        "loss_mode": loss_mode,
        "best_val": float(best_val),
        "model_state_dict": predictor.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    if extra is not None:
        payload.update(extra)
    torch.save(payload, path)


In [ ]:
def train_model(
    loss_mode: str,
    train_loader,
    val_loader,
    seed: int,
    lambdas=None,
    model_name: str = "NLL",
    architecture: str = "deepsets",
):
    set_seed(seed)

    predictor = build_predictor(
        architecture=architecture,
        slot_dim=64,
        num_slots=4,
        z_dim=64,
        hidden_dim=128,
        out_dim=10,
        dropout=0.0,
    ).to(device)

    opt = torch.optim.Adam(predictor.parameters(), lr=LR)

    ckpt_dir = os.path.join(CKPT_ROOT, f"seed_{seed}", model_name)
    os.makedirs(ckpt_dir, exist_ok=True)

    best_val = float("inf")
    best_state = None

    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_nll": []}

    for ep in range(1, NUM_EPOCHS + 1):
        tr = run_epoch(
            predictor,
            opt,
            train_loader,
            mode="train",
            loss_mode=loss_mode,
            lambdas=lambdas
        )

        va = run_epoch(
            predictor,
            opt,
            val_loader,
            mode="eval",
            loss_mode=loss_mode,
            lambdas=lambdas
        )

        va_nll = run_epoch(
            predictor,
            opt,
            val_loader,
            mode="eval",
            loss_mode="nll",
            lambdas=None
        )

        history["epoch"].append(ep)
        history["train_loss"].append(tr)
        history["val_loss"].append(va)
        history["val_nll"].append(va_nll)

        # For variance-regularization, select best checkpoint by validation NLL.
        criterion = va if loss_mode == "nll" else va_nll

        if (ep % CKPT_EVERY) == 0:
            save_checkpoint(
                path=os.path.join(ckpt_dir, f"epoch_{ep:04d}.pth"),
                predictor=predictor,
                optimizer=opt,
                epoch=ep,
                seed=seed,
                loss_mode=loss_mode,
                best_val=best_val,
                extra={
                    "lambdas": lambdas,
                    "lr": LR,
                    "architecture": architecture,
                }
            )

        if criterion < best_val:
            best_val = float(criterion)
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in predictor.state_dict().items()
            }

            save_checkpoint(
                path=os.path.join(ckpt_dir, "best.pth"),
                predictor=predictor,
                optimizer=opt,
                epoch=ep,
                seed=seed,
                loss_mode=loss_mode,
                best_val=best_val,
                extra={
                    "lambdas": lambdas,
                    "lr": LR,
                    "architecture": architecture,
                }
            )

        tqdm.write(
            f"[{model_name}] arch={architecture} seed={seed} epoch={ep:02d} "
            f"train={tr:.5f} val_obj={va:.5f} val_nll={va_nll:.5f} best={best_val:.5f}"
        )

    if best_state is None:
        raise RuntimeError("No best model state was saved. Check NUM_EPOCHS and validation loss.")

    predictor.load_state_dict(best_state)
    predictor.eval()

    return predictor, best_val, history

## **Per-factor Reg-ECE, per-factor coverage and per-factor PIT**

In [ ]:
def gaussian_interval_coverage_per_factor(mu, var, y, levels=(0.5,0.8,0.9,0.95)):
    sigma = np.sqrt(var + 1e-8)
    K = y.shape[1]
    out = np.zeros((K, len(levels)), dtype=np.float32)
    for li, p in enumerate(levels):
        z = norm.ppf((1+p)/2)
        lo = mu - z*sigma
        hi = mu + z*sigma
        out[:, li] = ((y >= lo) & (y <= hi)).mean(axis=0)
    return out

In [ ]:
def pit_values_per_factor(mu, var, y):
    sigma = np.sqrt(var + 1e-8)
    z = (y - mu) / sigma
    return norm.cdf(z)

In [ ]:
def evaluate_all(predictor, val_cal_loader, test_loader, seed: int, tag: str, test_latents_np=None, ood_q=0.1):
    mu_te, var_te, y_te = collect_mu_var(predictor, test_loader, T=None)
    rmse, mae = rmse_mae(mu_te, y_te)

    reg_ece_pf = reg_ece_sigma_mae(mu_te, var_te, y_te, per_factor=True)
    cov_pf = gaussian_interval_coverage_per_factor(
        mu_te, var_te, y_te, levels=(0.5, 0.8, 0.9, 0.95)
    )

    out = {
        "seed": seed,
        "model": tag,
        "rmse_mean": float(rmse.mean()),
        "mae_mean": float(mae.mean()),
        "nll": gaussian_nll_np(mu_te, var_te, y_te),
        "sharp": sharpness_np(var_te),
        "reg_ece": float(np.mean(reg_ece_pf)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf)),
        "reg_ece_pf": reg_ece_pf.tolist(),
        "coverage_pf_50_80_90_95": cov_pf.tolist(),
    }

    mu_cal, var_cal, y_cal = collect_mu_var(predictor, val_cal_loader, T=None)

    T = optimize_temperature(mu_cal, var_cal, y_cal)

    mu_teT, var_teT, y_teT = collect_mu_var(predictor, test_loader, T=T)
    rmseT, maeT = rmse_mae(mu_teT, y_teT)

    reg_ece_pf_T = reg_ece_sigma_mae(mu_teT, var_teT, y_teT, per_factor=True)
    cov_pf_T = gaussian_interval_coverage_per_factor(
        mu_teT, var_teT, y_teT, levels=(0.5, 0.8, 0.9, 0.95)
    )

    out_post = {
        "seed": seed,
        "model": tag + "+posthocT",
        "T": float(T),
        "rmse_mean": float(rmseT.mean()),
        "mae_mean": float(maeT.mean()),
        "nll": gaussian_nll_np(mu_teT, var_teT, y_teT),
        "sharp": sharpness_np(var_teT),
        "reg_ece": float(np.mean(reg_ece_pf_T)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf_T)),
        "reg_ece_pf": reg_ece_pf_T.tolist(),
        "coverage_pf_50_80_90_95": cov_pf_T.tolist(),
    }

    T_pf = optimize_temperature_per_factor(mu_cal, var_cal, y_cal)

    mu_te_pfT, var_te_pfT, y_te_pfT = collect_mu_var(predictor, test_loader, T=T_pf)
    rmse_pfT, mae_pfT = rmse_mae(mu_te_pfT, y_te_pfT)

    reg_ece_pf_pfT = reg_ece_sigma_mae(mu_te_pfT, var_te_pfT, y_te_pfT, per_factor=True)
    cov_pf_pfT = gaussian_interval_coverage_per_factor(
        mu_te_pfT, var_te_pfT, y_te_pfT, levels=(0.5, 0.8, 0.9, 0.95)
    )

    out_pfT = {
        "seed": seed,
        "model": tag + "+pfT",
        "rmse_mean": float(rmse_pfT.mean()),
        "mae_mean": float(mae_pfT.mean()),
        "nll": gaussian_nll_np(mu_te_pfT, var_te_pfT, y_te_pfT),
        "sharp": sharpness_np(var_te_pfT),
        "reg_ece": float(np.mean(reg_ece_pf_pfT)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf_pfT)),
        "reg_ece_pf": reg_ece_pf_pfT.tolist(),
        "coverage_pf_50_80_90_95": cov_pf_pfT.tolist(),
        "T_pf": T_pf.tolist(),
    }

    q_abs = conformal_q_abs_error(mu_cal, y_cal, alpha=0.1)
    lo_conf = mu_te - q_abs
    hi_conf = mu_te + q_abs
    cov_pf_conf = ((y_te >= lo_conf) & (y_te <= hi_conf)).mean(axis=0)

    out_conf = {
        "seed": seed,
        "model": tag + "+conformal90",
        "coverage_pf_mean": float(cov_pf_conf.mean()),
        "coverage_pf": cov_pf_conf.tolist(),
        "q_pf": q_abs.tolist(),
    }

    cdf_cal = fit_isotonic_cdf_calibrator(mu_cal, var_cal, y_cal)
    lo_cdf, hi_cdf = calibrated_interval_from_cdf(mu_te, var_te, cdf_cal, level=0.9)
    cov_pf_cdf = coverage_from_interval(lo_cdf, hi_cdf, y_te)

    out_cdf = {
        "seed": seed,
        "model": tag + "+cdfCal90",
        "rmse_mean": float(rmse.mean()),
        "mae_mean": float(mae.mean()),
        "nll": gaussian_nll_np(mu_te, var_te, y_te),
        "sharp": sharpness_np(var_te),
        "reg_ece": float(np.mean(reg_ece_pf)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf)),
        "reg_ece_pf": reg_ece_pf.tolist(),
        "coverage_pf_mean": float(cov_pf_cdf.mean()),
        "coverage_pf": cov_pf_cdf.tolist(),
    }

    lo_gsc, hi_gsc, qhat_pf = gaussian_split_conformal_interval(
        mu_cal, var_cal, y_cal,
        mu_te, var_te,
        alpha=0.1
    )
    cov_pf_gsc = coverage_from_interval(lo_gsc, hi_gsc, y_te)

    out_gsc = {
        "seed": seed,
        "model": tag + "+gaussSplitConf90",
        "rmse_mean": float(rmse.mean()),
        "mae_mean": float(mae.mean()),
        "nll": gaussian_nll_np(mu_te, var_te, y_te),
        "sharp": sharpness_np(var_te),
        "reg_ece": float(np.mean(reg_ece_pf)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf)),
        "reg_ece_pf": reg_ece_pf.tolist(),
        "coverage_pf_mean": float(cov_pf_gsc.mean()),
        "coverage_pf": cov_pf_gsc.tolist(),
        "qhat_pf": qhat_pf.tolist(),
    }

    ood_rows = []
    if test_latents_np is not None:
        ood_rows = eval_ood_slices_all_factors(mu_te, var_te, y_te, test_latents_np, q=ood_q)
        for r in ood_rows:
            r["seed"] = seed
            r["model"] = tag

    raw_pack = (mu_te, var_te, y_te)
    post_pack = (mu_teT, var_teT, y_teT)
    pfT_pack = (mu_te_pfT, var_te_pfT, y_te_pfT)

    return out, out_post, out_pfT, out_conf, out_cdf, out_gsc, ood_rows, raw_pack, post_pack, pfT_pack

# **OOD EVALUATION FUNCTIONS**

In [ ]:
@torch.no_grad()
def collect_mu_var_from_image_loader_frozen_sa(sa_model, predictor, image_loader, T=None, slot_init_seed=1234):
    sa_model.eval()
    predictor.eval()
    sa = sa_model.slot_attention

    mus, vars_, ys = [], [], []

    for x, y in tqdm(image_loader, desc="collect OOD (mu, var)"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        feats = sa_model.project(sa_model.encoder(x))
        K = sa.num_slots

        init_noise = make_fixed_slot_init_noise(
            feats=feats,
            num_slots=K,
            seed=slot_init_seed,
        )

        slots = sa(feats, init_noise=init_noise)

        mu, logvar = predictor(slots)
        var = torch.exp(logvar) + 1e-8

        if T is not None:
            if np.isscalar(T):
                var = var * (float(T) ** 2)
            else:
                Tt = torch.tensor(T, dtype=var.dtype, device=var.device).view(1, -1)
                var = var * (Tt ** 2)

        mus.append(mu.detach().cpu())
        vars_.append(var.detach().cpu())
        ys.append(y.detach().cpu())

    mu = torch.cat(mus, 0).numpy()
    var = torch.cat(vars_, 0).numpy()
    y = torch.cat(ys, 0).numpy()

    return mu, var, y

In [ ]:
def apply_temperature_to_var_np(var, temperature):
    if temperature is None:
        return var
    if np.isscalar(temperature):
        return var * (float(temperature) ** 2)
    temperature = np.asarray(temperature, dtype=np.float64).reshape(1, -1)
    return var * (temperature ** 2)


In [ ]:
def interval_width_stats(lower, upper):
    width = upper - lower
    return float(width.mean()), width.mean(axis=0).tolist()

In [ ]:
def predictive_metrics_from_pack(mu, var, y):
    rmse, mae = rmse_mae(mu, y)
    reg_ece_pf = reg_ece_sigma_mae(mu, var, y, per_factor=True)
    cov_pf = gaussian_interval_coverage_per_factor(
        mu, var, y, levels=(0.5, 0.8, 0.9, 0.95)
    )
    return {
        "rmse_mean": float(rmse.mean()),
        "mae_mean": float(mae.mean()),
        "nll": gaussian_nll_np(mu, var, y),
        "sharp": sharpness_np(var),
        "reg_ece": float(np.mean(reg_ece_pf)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf)),
        "reg_ece_pf": reg_ece_pf.tolist(),
        "coverage_pf_50_80_90_95": cov_pf.tolist(),
    }

In [ ]:
def build_predictive_ood_row(mu, var, y, seed, model_name, corruption, severity):
    row = {
        "seed": seed,
        "model": model_name,
        "variant_type": "predictive",
        "corruption": corruption,
        "severity": severity,
    }
    row.update(predictive_metrics_from_pack(mu, var, y))
    return row

In [ ]:
def build_interval_ood_row(mu, var, y, lower, upper, seed, model_name, corruption, severity, nominal_level=0.9):
    cov_pf = coverage_from_interval(lower, upper, y)
    width_mean, width_pf = interval_width_stats(lower, upper)

    rmse, mae = rmse_mae(mu, y)
    reg_ece_pf = reg_ece_sigma_mae(mu, var, y, per_factor=True)

    return {
        "seed": seed,
        "model": model_name,
        "variant_type": "interval",
        "corruption": corruption,
        "severity": severity,
        "nominal_level": nominal_level,
        "coverage_pf_mean": float(cov_pf.mean()),
        "coverage_pf": cov_pf.tolist(),
        "coverage_gap_abs_mean": float(np.mean(np.abs(cov_pf - nominal_level))),
        "interval_width_mean": width_mean,
        "interval_width_pf": width_pf,
        "rmse_mean": float(rmse.mean()),
        "mae_mean": float(mae.mean()),
        "nll": gaussian_nll_np(mu, var, y),
        "sharp": sharpness_np(var),
        "reg_ece": float(np.mean(reg_ece_pf)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf)),
        "reg_ece_pf": reg_ece_pf.tolist(),
    }

In [ ]:

def gaussian_split_conformal_qhat(mu_cal, var_cal, y_cal, alpha=0.1):
    """
    Compute per-factor Gaussian split conformal qhat.

    conformity score:
      score = max(lower_cal - y_cal, y_cal - upper_cal, 0)

    The zero term prevents negative conformity scores.
    """
    z = norm.ppf(1 - alpha / 2)
    sigma_cal = np.sqrt(var_cal + 1e-8)

    qlo_cal = mu_cal - z * sigma_cal
    qhi_cal = mu_cal + z * sigma_cal

    scores = np.maximum.reduce([
        qlo_cal - y_cal,
        y_cal - qhi_cal,
        np.zeros_like(y_cal)
    ])

    qhat = np.quantile(scores, 1 - alpha, axis=0)

    return qhat

In [ ]:
def apply_abs_error_conformal_interval(mu, q_abs):
    lower = mu - q_abs.reshape(1, -1)
    upper = mu + q_abs.reshape(1, -1)
    return lower, upper

In [ ]:
def apply_gaussian_split_conformal_interval(mu, var, qhat, alpha=0.1):
    z = norm.ppf(1 - alpha / 2)
    sigma = np.sqrt(var + 1e-8)

    qlo = mu - z * sigma
    qhi = mu + z * sigma

    lower = qlo - qhat.reshape(1, -1)
    upper = qhi + qhat.reshape(1, -1)
    return lower, upper

In [ ]:
def fit_ood_calibration_objects(predictor, val_cal_loader):
    mu_cal, var_cal, y_cal = collect_mu_var(predictor, val_cal_loader, T=None)

    temperature_scalar = optimize_temperature(mu_cal, var_cal, y_cal)
    temperature_per_factor = optimize_temperature_per_factor(mu_cal, var_cal, y_cal)

    q_abs_90 = conformal_q_abs_error(mu_cal, y_cal, alpha=0.1)
    cdf_calibrator_90 = fit_isotonic_cdf_calibrator(mu_cal, var_cal, y_cal)
    qhat_gsc_90 = gaussian_split_conformal_qhat(mu_cal, var_cal, y_cal, alpha=0.1)

    return {
        "temperature_scalar": temperature_scalar,
        "temperature_per_factor": temperature_per_factor,
        "q_abs_90": q_abs_90,
        "cdf_calibrator_90": cdf_calibrator_90,
        "qhat_gsc_90": qhat_gsc_90,
    }

In [ ]:
def evaluate_all_ood_variants_from_loader(
    sa_model,
    predictor,
    loader,
    seed,
    tag,
    corruption,
    severity,
    calib_objects,
):
    rows = []

    # raw OOD predictions: image -> frozen SA -> predictor
    mu_ood, var_ood, y_ood = collect_mu_var_from_image_loader_frozen_sa(
        sa_model=sa_model,
        predictor=predictor,
        image_loader=loader,
        T=None
    )

    # predictive variants
    rows.append(
        build_predictive_ood_row(
            mu_ood, var_ood, y_ood,
            seed=seed,
            model_name=tag,
            corruption=corruption,
            severity=severity
        )
    )

    var_ood_posthoc = apply_temperature_to_var_np(var_ood, calib_objects["temperature_scalar"])
    rows.append(
        build_predictive_ood_row(
            mu_ood, var_ood_posthoc, y_ood,
            seed=seed,
            model_name=f"{tag}+posthocT",
            corruption=corruption,
            severity=severity
        )
    )

    var_ood_pfT = apply_temperature_to_var_np(var_ood, calib_objects["temperature_per_factor"])
    rows.append(
        build_predictive_ood_row(
            mu_ood, var_ood_pfT, y_ood,
            seed=seed,
            model_name=f"{tag}+pfT",
            corruption=corruption,
            severity=severity
        )
    )

    # interval variants
    lower_conf, upper_conf = apply_abs_error_conformal_interval(mu_ood, calib_objects["q_abs_90"])
    rows.append(
        build_interval_ood_row(
            mu_ood, var_ood, y_ood,
            lower_conf, upper_conf,
            seed=seed,
            model_name=f"{tag}+conformal90",
            corruption=corruption,
            severity=severity,
            nominal_level=0.9
        )
    )

    lower_cdf, upper_cdf = calibrated_interval_from_cdf(
        mu_ood, var_ood, calib_objects["cdf_calibrator_90"], level=0.9
    )
    rows.append(
        build_interval_ood_row(
            mu_ood, var_ood, y_ood,
            lower_cdf, upper_cdf,
            seed=seed,
            model_name=f"{tag}+cdfCal90",
            corruption=corruption,
            severity=severity,
            nominal_level=0.9
        )
    )

    lower_gsc, upper_gsc = apply_gaussian_split_conformal_interval(
        mu_ood, var_ood, calib_objects["qhat_gsc_90"], alpha=0.1
    )
    rows.append(
        build_interval_ood_row(
            mu_ood, var_ood, y_ood,
            lower_gsc, upper_gsc,
            seed=seed,
            model_name=f"{tag}+gaussSplitConf90",
            corruption=corruption,
            severity=severity,
            nominal_level=0.9
        )
    )

    return rows

# **PLOT FUNCTIONS**

In [ ]:
def plot_loss_curves(all_histories):
    # Validation
    plt.figure(figsize=(7,4))
    for h in all_histories:
        plt.plot(h["epoch"], h["val_loss"], marker="o", label=f"{h['mode']} val (seed {h['seed']})")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Validation loss curves")
    plt.grid(True); plt.legend()
    #plt.show()

    # Training
    plt.figure(figsize=(7,4))
    for h in all_histories:
        plt.plot(h["epoch"], h["train_loss"], marker="o", label=f"{h['mode']} train (seed {h['seed']})")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training loss curves")
    plt.grid(True); plt.legend()
    #plt.show()

In [ ]:
def gaussian_interval_coverage(mu, var, y, levels=(0.5,0.8,0.9,0.95)):
    sigma = np.sqrt(var + 1e-8)
    covs = []
    for p in levels:
        z = norm.ppf((1+p)/2)
        lo = mu - z*sigma
        hi = mu + z*sigma
        covs.append(float(((y>=lo) & (y<=hi)).mean()))
    return np.array(covs)

In [ ]:
def pit_values(mu, var, y):
    sigma = np.sqrt(var + 1e-8)
    z = (y - mu) / sigma
    return norm.cdf(z).reshape(-1)

In [ ]:
def plot_coverage_curve(results_dict, levels=(0.5,0.8,0.9,0.95), title="Coverage curve"):
    xs = np.array(levels)
    plt.figure(figsize=(5.5,5))
    for label, (mu, var, y) in results_dict.items():
        emp = gaussian_interval_coverage(mu, var, y, levels=levels)
        plt.plot(xs, emp, marker="o", label=label)
    plt.plot(xs, xs, linestyle="--", label="Ideal")
    plt.xlabel("Nominal coverage"); plt.ylabel("Empirical coverage")
    plt.title(title); plt.grid(True); plt.legend()
    #plt.show()

In [ ]:
def plot_pit_histograms(results_dict, bins=20, title="PIT histograms"):
    n = len(results_dict)
    cols = 2
    rows = math.ceil(n/cols)
    plt.figure(figsize=(7, 3.2*rows))
    for i, (label, (mu,var,y)) in enumerate(results_dict.items(), 1):
        pit = pit_values(mu,var,y)
        ax = plt.subplot(rows, cols, i)
        ax.hist(pit, bins=bins, range=(0,1), density=True)
        ax.set_title(label); ax.set_xlabel("PIT"); ax.set_ylabel("Density")
        ax.grid(True)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    #plt.show()

**PER-FACTOR PLOTS**

In [ ]:
def factor_coverage_table(mu, var, y, levels=(0.5, 0.8, 0.9, 0.95)):
    cov_pf = gaussian_interval_coverage_per_factor(mu, var, y, levels=levels)  # (K, L)
    df_cov = pd.DataFrame(cov_pf, columns=[f"cov@{p}" for p in levels])
    df_cov.insert(0, "factor", FACTOR_NAMES[:df_cov.shape[0]])
    return df_cov

def factor_reg_ece_table(mu, var, y, num_bins=15):
    ece_pf = reg_ece_sigma_mae(mu, var, y, num_bins=num_bins, per_factor=True)
    df_ece = pd.DataFrame({"factor": FACTOR_NAMES[:len(ece_pf)], "reg_ece": ece_pf})
    df_ece = df_ece.sort_values("reg_ece", ascending=False).reset_index(drop=True)
    return df_ece

def plot_per_factor_coverage_curves(results_dict, levels=(0.5,0.8,0.9,0.95), save_path=None, title="Per-factor coverage"):
    """
    Plot One figure per factor (10 figures) in a 5 x 2 grid
    This makes a grid: rows=5, cols=2 (for K=10), each subplot shows coverage curve for that factor.
    """
    xs = np.array(levels)
    any_label = next(iter(results_dict.keys()))
    K = results_dict[any_label][0].shape[1]

    cols = 2
    rows = int(math.ceil(K / cols))
    plt.figure(figsize=(10, 3.2 * rows))

    for k in range(K):
        ax = plt.subplot(rows, cols, k + 1)
        for label, (mu, var, y) in results_dict.items():
            cov_pf = gaussian_interval_coverage_per_factor(mu, var, y, levels=levels)  # (K,L)
            ax.plot(xs, cov_pf[k], marker="o", label=label)
        ax.plot(xs, xs, linestyle="--", label="Ideal")
        ax.set_title(f"Coverage: {FACTOR_NAMES[k]}")
        ax.set_xlabel("Nominal"); ax.set_ylabel("Empirical")
        ax.grid(True)

        if k == 0:
            ax.legend()

    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)
    #plt.show()

def plot_per_factor_pit_histograms(results_dict, bins=20, save_path=None, title="Per-factor PIT"):
    """
    For each model, plot PIT histograms for all factors (grid).
    """
    any_label = next(iter(results_dict.keys()))
    K = results_dict[any_label][0].shape[1]

    for label, (mu, var, y) in results_dict.items():
        pit_pf = pit_values_per_factor(mu, var, y)  # (N,K)

        cols = 2
        rows = int(math.ceil(K / cols))
        plt.figure(figsize=(10, 3.2 * rows))

        for k in range(K):
            ax = plt.subplot(rows, cols, k + 1)
            ax.hist(pit_pf[:, k], bins=bins, range=(0,1), density=True)
            ax.set_title(f"{label} PIT: {FACTOR_NAMES[k]}")
            ax.set_xlabel("PIT"); ax.set_ylabel("Density")
            ax.grid(True)

        plt.suptitle(f"{title} — {label}", y=1.02)
        plt.tight_layout()

        if save_path is not None:
            base, ext = os.path.splitext(save_path)
            sp = f"{base}_{label.replace('/', '_').replace(' ', '_')}{ext}"
            plt.savefig(sp, dpi=200, bbox_inches="tight")
            print("Saved:", sp)

        #plt.show()

def show_worst_factors(mu, var, y, levels=(0.9,), topk=3):
    """
    Returns a table listing worst factors by:
      - reg_ece
      - |coverage(level) - level|
    """
    df_ece = factor_reg_ece_table(mu, var, y)
    out = {"worst_reg_ece": df_ece.head(topk)}

    for lv in levels:
        cov_pf = gaussian_interval_coverage_per_factor(mu, var, y, levels=(lv,))[:, 0]
        err = np.abs(cov_pf - lv)
        df_cov = pd.DataFrame({"factor": FACTOR_NAMES[:len(cov_pf)], "cov": cov_pf, "abs_gap": err})
        df_cov = df_cov.sort_values("abs_gap", ascending=False).reset_index(drop=True)
        out[f"worst_cov_gap@{lv}"] = df_cov.head(topk)

    return out


In [ ]:
def build_results_dict_from_packs(raw_pack, post_pack, pfT_pack, tag):
    return {
        tag: raw_pack,
        tag + "+posthocT": post_pack,
        tag + "+pfT": pfT_pack,
    }

In [ ]:

def save_per_seed_plots(seed, tag, raw_pack, post_pack, pfT_pack, seed_log_dir):
    os.makedirs(seed_log_dir, exist_ok=True)
    results_seed = build_results_dict_from_packs(raw_pack, post_pack, pfT_pack, tag)

    xs = np.array((0.5, 0.8, 0.9, 0.95))
    plt.figure(figsize=(5.5, 5))
    for label, (mu, var, y) in results_seed.items():
        emp = gaussian_interval_coverage(mu, var, y, levels=(0.5, 0.8, 0.9, 0.95))
        plt.plot(xs, emp, marker="o", label=label)
    plt.plot(xs, xs, linestyle="--", label="Ideal")
    plt.xlabel("Nominal coverage")
    plt.ylabel("Empirical coverage")
    plt.title(f"Coverage Curve (Seed {seed}, {tag})")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(seed_log_dir, f"coverage_curve_{tag}_seed{seed}.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close()

    n = len(results_seed)
    cols = 2
    rows = math.ceil(n / cols)
    plt.figure(figsize=(7, 3.2 * rows))
    for i, (label, (mu, var, y)) in enumerate(results_seed.items(), 1):
        pit = pit_values(mu, var, y)
        ax = plt.subplot(rows, cols, i)
        ax.hist(pit, bins=20, range=(0, 1), density=True)
        ax.set_title(label)
        ax.set_xlabel("PIT")
        ax.set_ylabel("Density")
        ax.grid(True)
    plt.suptitle(f"PIT Histograms (Seed {seed}, {tag})", y=1.02)
    plt.tight_layout()
    plt.savefig(
        os.path.join(seed_log_dir, f"pit_histograms_{tag}_seed{seed}.png"),
        dpi=200,
        bbox_inches="tight"
    )
    plt.close()

    plot_per_factor_coverage_curves(
        results_seed,
        levels=(0.5, 0.8, 0.9, 0.95),
        save_path=os.path.join(seed_log_dir, f"per_factor_coverage_{tag}_seed{seed}.png"),
        title=f"Per-Factor Coverage (Seed {seed}, {tag})"
    )

    plot_per_factor_pit_histograms(
        results_seed,
        bins=20,
        save_path=os.path.join(seed_log_dir, f"per_factor_pit_{tag}_seed{seed}.png"),
        title=f"Per-Factor PIT (Seed {seed}, {tag})"
    )

In [ ]:
def plot_coverage_curve_over_seeds(plot_cache_all_seeds, levels=(0.5, 0.8, 0.9, 0.95), save_path=None):
    xs = np.array(levels)
    plt.figure(figsize=(6, 5))

    for label, packs in plot_cache_all_seeds.items():
        per_seed_cov = []
        for mu, var, y in packs:
            per_seed_cov.append(gaussian_interval_coverage(mu, var, y, levels=levels))
        per_seed_cov = np.stack(per_seed_cov, axis=0)

        mean_cov = per_seed_cov.mean(axis=0)
        lo_cov, hi_cov = [], []
        for j in range(per_seed_cov.shape[1]):
            m, lo, hi = mean_ci(per_seed_cov[:, j])
            lo_cov.append(lo)
            hi_cov.append(hi)
        lo_cov = np.array(lo_cov)
        hi_cov = np.array(hi_cov)

        plt.plot(xs, mean_cov, marker="o", label=label)
        plt.fill_between(xs, lo_cov, hi_cov, alpha=0.20)

    plt.plot(xs, xs, linestyle="--", color="black", label="Ideal")
    plt.xlabel("Nominal coverage")
    plt.ylabel("Empirical coverage")
    plt.title(f"Coverage over seeds (mean ± 95% CI, n={len(SEEDS)})")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)
    #plt.show()

In [ ]:
def plot_reg_ece_over_seeds(df_metrics, save_path=None):
    models = sorted(df_metrics["model"].unique())
    means, los, his = [], [], []

    for model in models:
        vals = df_metrics[df_metrics["model"] == model]["reg_ece"].values
        m, lo, hi = mean_ci(vals)
        means.append(m)
        los.append(m - lo)
        his.append(hi - m)

    x = np.arange(len(models))
    plt.figure(figsize=(8, 4.5))
    plt.bar(x, means, yerr=[los, his], capsize=4)
    plt.xticks(x, models, rotation=30, ha="right")
    plt.ylabel("Reg-ECE")
    plt.title(f"Reg-ECE over seeds (mean ± 95% CI, n={len(SEEDS)})")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)
    #plt.show()

In [ ]:
def plot_nll_over_seeds(df_metrics, save_path=None):
    models = sorted(df_metrics["model"].unique())
    means, los, his = [], [], []

    for model in models:
        vals = df_metrics[df_metrics["model"] == model]["nll"].values
        m, lo, hi = mean_ci(vals)
        means.append(m)
        los.append(m - lo)
        his.append(hi - m)

    x = np.arange(len(models))
    plt.figure(figsize=(8, 4.5))
    plt.bar(x, means, yerr=[los, his], capsize=4)
    plt.xticks(x, models, rotation=30, ha="right")
    plt.ylabel("NLL")
    plt.title(f"NLL over seeds (mean ± 95% CI, n={len(SEEDS)})")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print("Saved:", save_path)
    #plt.show()

In [ ]:
def aggregate_invariance_table(df_stability_local):
    # Aggregate matched-pair stability and pair-quality diagnostics over seeds.
    metric_cols = [
        "inv_non_j_mu",
        "sens_j_mu",
        "inv_z",
        "n_pairs_used",

        "pair_n_raw_candidates_considered",
        "pair_n_pass_target_gap",
        "pair_n_pass_protected_max_drift",
        "pair_n_pass_joint_protected_dist",
        "pair_n_passing_image_slot_mask_checks",
        "pair_n_finally_selected",

        "pair_target_gap_std_mean",
        "pair_target_gap_std_median",
        "pair_target_gap_std_min",

        "pair_protected_joint_dist_std_mean",
        "pair_protected_joint_dist_std_median",

        "pair_protected_max_drift_std_mean",
        "pair_protected_max_drift_std_median",
        "pair_protected_max_drift_std_max",

        "pair_mask_iou_mean",
        "pair_mask_iou_median",
        "pair_mask_iou_min_mean",

        "pair_slot_l2_mean",
        "pair_slot_l2_median",
        "pair_slot_l2_max",

        "pair_image_mse_mean",
        "pair_image_mse_median",

        "pair_mask_entropy_mean",
        "pair_mask_entropy_min_mean",
        "pair_mask_entropy_max_mean",
        "pair_mask_entropy_std_mean",

        "pair_mask_area_mean",
        "pair_mask_area_min_mean",
        "pair_mask_area_max_mean",
        "pair_mask_area_std_mean",

        "pair_mask_within_image_diversity_mean",
    ]

    rows_out = []

    for (model, factor_j), sub in df_stability_local.groupby(["model", "factor_j"]):
        factor_j = int(factor_j)

        row = {
            "model": model,
            "factor_j": factor_j,
            "factor_name": FACTOR_NAMES[factor_j],
            "n_seeds": sub["seed"].nunique(),
            "n_zero_pair_seeds": int((sub["n_pairs_used"] == 0).sum()),
        }

        for c in metric_cols:
            if c not in sub.columns:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_ci95_lo"] = np.nan
                row[f"{c}_ci95_hi"] = np.nan
                continue

            vals = sub[c].dropna().values

            if len(vals) >= 2:
                m, lo, hi = mean_ci(vals)
            elif len(vals) == 1:
                m = lo = hi = float(vals[0])
            else:
                m = lo = hi = np.nan

            row[f"{c}_mean"] = m
            row[f"{c}_ci95_lo"] = lo
            row[f"{c}_ci95_hi"] = hi

        rows_out.append(row)

    return pd.DataFrame(rows_out).sort_values(["model", "factor_j"])

## **OOD / shift function**

In [ ]:
def slice_indices_by_factor_quantile(latents_np, factor_j, q_lo=0.9, q_hi=1.0):
    x = latents_np[:, factor_j]
    lo = np.quantile(x, q_lo)
    hi = np.quantile(x, q_hi)
    return np.where((x >= lo) & (x <= hi))[0]

In [ ]:
def eval_metrics_on_subset(mu, var, y, idx, levels=(0.5,0.8,0.9,0.95)):
    mu_s = mu[idx]
    var_s = var[idx]
    y_s = y[idx]

    rmse, mae = rmse_mae(mu_s, y_s)
    reg_ece_pf = reg_ece_sigma_mae(mu_s, var_s, y_s, per_factor=True)
    cov_pf = gaussian_interval_coverage_per_factor(mu_s, var_s, y_s, levels=levels)

    out = {
        "n": int(len(idx)),
        "rmse_mean": float(rmse.mean()),
        "mae_mean": float(mae.mean()),
        "nll": gaussian_nll_np(mu_s, var_s, y_s),
        "sharp": sharpness_np(var_s),
        "reg_ece": float(np.mean(reg_ece_pf)),
        "reg_ece_pf_mean": float(np.mean(reg_ece_pf)),
        "reg_ece_pf": reg_ece_pf.tolist(),
        "coverage_pf_50_80_90_95": cov_pf.tolist(),
    }
    return out

In [ ]:

def eval_ood_slices_all_factors(mu, var, y, latents_np, q=0.1, levels=(0.5,0.8,0.9,0.95)):
    """
    OOD slices for ALL factors:
      - low slice  [0, q]
      - high slice [1-q, 1]
    Returns list of rows (one per factor per slice).
    """
    rows = []
    K = latents_np.shape[1]

    for j in range(K):
        idx_lo = slice_indices_by_factor_quantile(latents_np, j, q_lo=0.0, q_hi=q)
        idx_hi = slice_indices_by_factor_quantile(latents_np, j, q_lo=1.0 - q, q_hi=1.0)

        m_lo = eval_metrics_on_subset(mu, var, y, idx_lo, levels=levels)
        m_hi = eval_metrics_on_subset(mu, var, y, idx_hi, levels=levels)

        rows.append({"factor_j": j, "slice": f"lo{int(q*100)}", **m_lo})
        rows.append({"factor_j": j, "slice": f"hi{int(q*100)}", **m_hi})

    return rows


# **RUN FOR ALL SEEDS**

In [ ]:
def mean_ci(values, alpha=0.05):
    x = np.asarray(values, dtype=np.float64)
    x = x[~np.isnan(x)]

    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan

    m = float(x.mean())

    if n == 1:
        return m, m, m

    s = float(x.std(ddof=1))
    h = float(t.ppf(1 - alpha / 2, df=n - 1) * s / np.sqrt(n))
    return m, m - h, m + h

In [ ]:
def add_ci_table(df, metric_cols=("rmse_mean", "mae_mean", "nll", "sharp", "reg_ece")):
    out_rows = []

    for model in sorted(df["model"].unique()):
        sub = df[df["model"] == model]
        row = {"model": model, "n_seeds": sub["seed"].nunique()}

        for c in metric_cols:
            if c not in sub.columns:
                row[f"{c}_mean"] = np.nan
                row[f"{c}_ci95_lo"] = np.nan
                row[f"{c}_ci95_hi"] = np.nan
                continue

            m, lo, hi = mean_ci(sub[c].values)
            row[f"{c}_mean"] = m
            row[f"{c}_ci95_lo"] = lo
            row[f"{c}_ci95_hi"] = hi

        out_rows.append(row)

    return pd.DataFrame(out_rows)

In [ ]:
def build_pair_cache_for_seed(
    seed,
    seed_log_dir,
    sa_model,
    test_ds,
    config,
    make_visuals=True,
):
    # Build matched pairs once per seed and factor.
    # Pair construction does not depend on the downstream predictor.
    pair_cache = {}
    pair_source = "matched_latent_visual_slot"

    pair_root_dir = os.path.join(seed_log_dir, "matched_pairs", "_PAIR_CACHE")
    selected_pair_dir = os.path.join(pair_root_dir, "selected_pairs")
    accepted_pair_dir = os.path.join(pair_root_dir, "accepted_pairs")
    rejected_pair_dir = os.path.join(pair_root_dir, "rejected_pairs")
    pair_vis_dir = os.path.join(pair_root_dir, "pair_visual_examples")

    os.makedirs(selected_pair_dir, exist_ok=True)
    os.makedirs(accepted_pair_dir, exist_ok=True)
    os.makedirs(rejected_pair_dir, exist_ok=True)
    os.makedirs(pair_vis_dir, exist_ok=True)

    seed_pair_rows_cache = []

    for factor_j in range(10):
        print("\n" + "-" * 80)
        print(f"BUILDING PAIR CACHE | seed={seed} | factor={factor_j}")

        pairs_found, selected_rows, accepted_rows, rejected_rows, pair_diag = build_matched_pairs(
            latents_np=test_ds.latents,
            factor_j=factor_j,
            sa_model=sa_model,
            dataset=test_ds,
            protected_factors=PROTECTED_FACTORS[factor_j],
            seed=seed,
            config=config,
            verbose=True,
        )

        pair_cache[factor_j] = {
            "pairs_found": pairs_found,
            "selected_rows": selected_rows,
            "accepted_rows": accepted_rows,
            "rejected_rows": rejected_rows,
            "pair_diag": pair_diag,
            "pair_source": pair_source,
        }

        pd.DataFrame(selected_rows).to_csv(
            os.path.join(selected_pair_dir, f"selected_pairs_f{factor_j}_seed{seed}.csv"),
            index=False
        )

        pd.DataFrame(accepted_rows).to_csv(
            os.path.join(accepted_pair_dir, f"accepted_pairs_f{factor_j}_seed{seed}.csv"),
            index=False
        )

        pd.DataFrame(rejected_rows).to_csv(
            os.path.join(rejected_pair_dir, f"rejected_pairs_f{factor_j}_seed{seed}.csv"),
            index=False
        )

        pair_row = {
            "seed": seed,
            "model": "_PAIR_CACHE",
            "architecture": "_PAIR_CACHE",
            "architecture_tag": "_PAIR_CACHE",
            "loss_mode": "_PAIR_CACHE",
            "loss_tag": "_PAIR_CACHE",
            "factor_j": factor_j,
            "factor_name": FACTOR_NAMES[factor_j],
            "pair_source": pair_source,
            "n_pairs_found": len(pairs_found),
            "protected_factors": str(PROTECTED_FACTORS[factor_j]),
            **pair_diag,
        }

        seed_pair_rows_cache.append(pair_row)

        if make_visuals:
            if len(selected_rows) > 0:
                plot_pair_examples_from_rows(
                    dataset=test_ds,
                    rows=selected_rows,
                    factor_j=factor_j,
                    title=f"Accepted pairs | PAIR_CACHE | seed={seed} | factor={factor_j}",
                    n_pairs=min(4, len(selected_rows)),
                    save_path=os.path.join(pair_vis_dir, f"accepted_f{factor_j}_seed{seed}.png"),
                )

                plot_pair_mask_grids_from_rows(
                    sa_model=sa_model,
                    dataset=test_ds,
                    rows=selected_rows,
                    factor_j=factor_j,
                    title=f"Accepted mask grids | PAIR_CACHE | seed={seed} | factor={factor_j}",
                    n_pairs=min(4, len(selected_rows)),
                    save_path=os.path.join(pair_vis_dir, f"accepted_masks_f{factor_j}_seed{seed}.png"),
                )

            if len(rejected_rows) > 0:
                plot_pair_examples_from_rows(
                    dataset=test_ds,
                    rows=rejected_rows,
                    factor_j=factor_j,
                    title=f"Rejected pairs | PAIR_CACHE | seed={seed} | factor={factor_j}",
                    n_pairs=min(4, len(rejected_rows)),
                    save_path=os.path.join(pair_vis_dir, f"rejected_f{factor_j}_seed{seed}.png"),
                )

                plot_pair_mask_grids_from_rows(
                    sa_model=sa_model,
                    dataset=test_ds,
                    rows=rejected_rows,
                    factor_j=factor_j,
                    title=f"Rejected mask grids | PAIR_CACHE | seed={seed} | factor={factor_j}",
                    n_pairs=min(4, len(rejected_rows)),
                    save_path=os.path.join(pair_vis_dir, f"rejected_masks_f{factor_j}_seed{seed}.png"),
                )

    pd.DataFrame(seed_pair_rows_cache).to_csv(
        os.path.join(seed_log_dir, f"pair_search_CACHE_seed{seed}.csv"),
        index=False
    )

    return pair_cache, seed_pair_rows_cache

In [ ]:
# RUN ALL SEEDS

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_ROOT, exist_ok=True)

def make_experiment_name(arch_tag, loss_tag):
    return f"{arch_tag}_{loss_tag}"

def make_run_id(seed, arch_tag, loss_tag):
    return f"{arch_tag}_{loss_tag}_seed{seed}"

def make_run_dirs(log_root, seed, arch_tag, loss_tag):
    experiment_name = make_experiment_name(arch_tag, loss_tag)
    run_id = make_run_id(seed, arch_tag, loss_tag)

    base_dir = os.path.join(
        log_root,
        "by_architecture",
        arch_tag,
        loss_tag,
        f"seed_{seed}"
    )

    dirs = {
        "base": base_dir,
        "metrics": os.path.join(base_dir, "metrics"),
        "plots": os.path.join(base_dir, "plots"),
        "ood": os.path.join(base_dir, "ood"),
        "matched_pairs": os.path.join(base_dir, "matched_pairs"),
        "selected_pairs": os.path.join(base_dir, "matched_pairs", "selected_pairs_from_cache"),
        "accepted_pairs": os.path.join(base_dir, "matched_pairs", "accepted_pairs_from_cache"),
        "rejected_pairs": os.path.join(base_dir, "matched_pairs", "rejected_pairs_from_cache"),
        "identifiability": os.path.join(base_dir, "identifiability"),
        "attention": os.path.join(base_dir, "attention"),
    }

    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    return dirs, experiment_name, run_id


rows = []
rows_conf = []
rows_ood = []
rows_ood_shift = []
rows_stability = []
rows_r2 = []
rows_pair_search = []
rows_factor_attention = []
all_histories = []

plot_cache_all_seeds = defaultdict(list)

# Predictor architectures

ARCHITECTURE_EXPERIMENTS = [
    # ("meanpool_linear", "MeanPoolLinear", "nll", "NLL", None),
    # ("meanpool",        "MeanPoolMLP",    "nll", "NLL", None),
    ("flatmlp",         "FlatMLP",        "nll", "NLL", None),
    # ("deepsets",        "DeepSets",       "nll", "NLL", None),
    # ("factorquery",     "FactorQuery",    "nll", "NLL", None),

    #("meanpool_linear", "MeanPoolLinear", "residual_variance_reg", "RESVAR", {"lambda_var_reg": 1.0, "lambda_l1": 1e-4}),
    #("meanpool",        "MeanPoolMLP",    "residual_variance_reg", "RESVAR", {"lambda_var_reg": 1.0, "lambda_l1": 1e-4}),
    ("flatmlp",         "FlatMLP",        "residual_variance_reg", "RESVAR", {"lambda_var_reg": 1.0, "lambda_l1": 1e-4}),
    #("deepsets",        "DeepSets",       "residual_variance_reg", "RESVAR", {"lambda_var_reg": 1.0, "lambda_l1": 1e-4}),
    #("factorquery",     "FactorQuery",    "residual_variance_reg", "RESVAR", {"lambda_var_reg": 1.0, "lambda_l1": 1e-4}),
]

# use {1,3,5} for severities

OOD_CORRUPTIONS = [
    ("gaussian_blur", [5]),
    ("brightness", [5]),
    ("gaussian_noise", [5]),
]

for seed in SEEDS:
    print("\n" + "=" * 90)
    print(f"SEED: {seed}")

    seed_log_dir = os.path.join(LOG_ROOT, f"seed_{seed}")
    os.makedirs(seed_log_dir, exist_ok=True)

    seed_cache_dir = os.path.join(LOG_ROOT, "pair_cache", f"seed_{seed}")
    os.makedirs(seed_cache_dir, exist_ok=True)

    train_loader, val_sel_loader, val_cal_loader, test_loader, S_tr, Y_tr, S_te, Y_te = make_loaders(seed)

    print(
        f"Sizes | train={len(train_loader.dataset)} | "
        f"val_sel={len(val_sel_loader.dataset)} | "
        f"val_cal={len(val_cal_loader.dataset)} | "
        f"test={len(test_loader.dataset)}"
    )

    pair_cache, seed_pair_rows_cache = build_pair_cache_for_seed(
        seed=seed,
        seed_log_dir=seed_cache_dir,
        sa_model=sa_model,
        test_ds=test_ds,
        config=PAIR_CONFIG,
        make_visuals=True,
    )

    for r in seed_pair_rows_cache:
        r["seed"] = seed
        r["pair_source_scope"] = "shared_pair_cache"

    rows_pair_search.extend(seed_pair_rows_cache)

    pd.DataFrame(seed_pair_rows_cache).to_csv(
        os.path.join(seed_cache_dir, f"pair_search_cache_seed{seed}.csv"),
        index=False
    )

    for architecture, arch_tag, loss_mode, loss_tag, lambdas in ARCHITECTURE_EXPERIMENTS:
        run_dirs, experiment_name, run_id = make_run_dirs(
            log_root=LOG_ROOT,
            seed=seed,
            arch_tag=arch_tag,
            loss_tag=loss_tag,
        )

        tag = experiment_name

        experiment_meta = {
            "experiment_name": experiment_name,
            "run_id": run_id,
            "architecture": architecture,
            "architecture_tag": arch_tag,
            "loss_mode": loss_mode,
            "loss_tag": loss_tag,
        }

        print("\n" + "-" * 80)
        print(f"MODEL: {tag} | ARCH: {architecture} | LOSS: {loss_mode} | SEED: {seed}")

        predictor, best_val, hist = train_model(
            loss_mode=loss_mode,
            train_loader=train_loader,
            val_loader=val_sel_loader,
            seed=seed,
            lambdas=lambdas,
            model_name=tag,
            architecture=architecture,
        )

        hist_out = {
            "epoch": hist["epoch"],
            "train_loss": hist["train_loss"],
            "val_loss": hist["val_loss"],
            "val_nll": hist["val_nll"],
            "seed": seed,
            "mode": tag,
            "model": tag,
            **experiment_meta,
        }

        all_histories.append(hist_out)

        hist_rows = []

        for i, ep in enumerate(hist["epoch"]):
            hist_rows.append({
                "seed": seed,
                "model": tag,
                "epoch": ep,
                "train_loss": hist["train_loss"][i],
                "val_loss": hist["val_loss"][i],
                "val_nll": hist["val_nll"][i],
                "best_val": best_val,
                **experiment_meta,
            })

        pd.DataFrame(hist_rows).to_csv(
            os.path.join(run_dirs["metrics"], f"history_{tag}_seed{seed}.csv"),
            index=False
        )

        out, out_post, out_pfT, out_conf, out_cdf, out_gsc, ood_rows_local, raw_pack, post_pack, pfT_pack = evaluate_all(
            predictor=predictor,
            val_cal_loader=val_cal_loader,
            test_loader=test_loader,
            seed=seed,
            tag=tag,
            test_latents_np=test_ds.latents,
            ood_q=0.1,
        )

        for d in [out, out_post, out_pfT, out_conf, out_cdf, out_gsc]:
            d.update(experiment_meta)

        for r in ood_rows_local:
            r.update(experiment_meta)

        rows += [out, out_post, out_pfT, out_cdf, out_gsc]
        rows_conf.append(out_conf)
        rows_ood += ood_rows_local

        pd.DataFrame([out, out_post, out_pfT, out_cdf, out_gsc]).to_csv(
            os.path.join(run_dirs["metrics"], f"metrics_{tag}_seed{seed}.csv"),
            index=False
        )

        pd.DataFrame([out_conf]).to_csv(
            os.path.join(run_dirs["metrics"], f"conformal_{tag}_seed{seed}.csv"),
            index=False
        )

        pd.DataFrame(ood_rows_local).to_csv(
            os.path.join(run_dirs["ood"], f"ood_slices_{tag}_seed{seed}.csv"),
            index=False
        )

        calib_objects = fit_ood_calibration_objects(predictor, val_cal_loader)

        ood_shift_rows_local = []

        for corruption_name, severity_list in OOD_CORRUPTIONS:
            for severity in severity_list:
                print(f"[{tag}] seed={seed} | OOD corruption={corruption_name} severity={severity}")

                _, ood_loader = build_ood_loader(
                    root=test_root,
                    corruption=corruption_name,
                    severity=severity,
                    batch_size=BATCH_SIZE,
                    image_size=64,
                    num_workers=NUM_WORKERS,
                    noise_base_seed=100000 + seed,
                )

                ood_variant_rows = evaluate_all_ood_variants_from_loader(
                    sa_model=sa_model,
                    predictor=predictor,
                    loader=ood_loader,
                    seed=seed,
                    tag=tag,
                    corruption=corruption_name,
                    severity=severity,
                    calib_objects=calib_objects,
                )

                for r in ood_variant_rows:
                    r.update(experiment_meta)

                ood_shift_rows_local.extend(ood_variant_rows)

        rows_ood_shift += ood_shift_rows_local

        pd.DataFrame(ood_shift_rows_local).to_csv(
            os.path.join(run_dirs["ood"], f"ood_shift_{tag}_seed{seed}.csv"),
            index=False
        )

        plot_cache_all_seeds[tag].append(raw_pack)
        plot_cache_all_seeds[tag + "+posthocT"].append(post_pack)
        plot_cache_all_seeds[tag + "+pfT"].append(pfT_pack)

        save_per_seed_plots(
            seed=seed,
            tag=tag,
            raw_pack=raw_pack,
            post_pack=post_pack,
            pfT_pack=pfT_pack,
            seed_log_dir=run_dirs["plots"],
        )

        seed_stability_rows = []
        seed_pair_rows_model = []

        for factor_j in range(10):
            cached = pair_cache[factor_j]

            selected_pairs = cached["pairs_found"]
            selected_rows = cached["selected_rows"]
            accepted_rows = cached["accepted_rows"]
            rejected_rows = cached["rejected_rows"]
            pair_diag = cached["pair_diag"]
            pair_source = cached["pair_source"]

            pd.DataFrame(selected_rows).to_csv(
                os.path.join(
                    run_dirs["selected_pairs"],
                    f"selected_pairs_{tag}_f{factor_j}_seed{seed}.csv"
                ),
                index=False
            )

            pd.DataFrame(accepted_rows).to_csv(
                os.path.join(
                    run_dirs["accepted_pairs"],
                    f"accepted_pairs_{tag}_f{factor_j}_seed{seed}.csv"
                ),
                index=False
            )

            pd.DataFrame(rejected_rows).to_csv(
                os.path.join(
                    run_dirs["rejected_pairs"],
                    f"rejected_pairs_{tag}_f{factor_j}_seed{seed}.csv"
                ),
                index=False
            )

            ci = evaluate_predictor_matched_pair_stability(
                predictor=predictor,
                slots_cpu=S_te,
                latents_np=test_ds.latents,
                factor_j=factor_j,
                pairs=selected_pairs,
                pair_source=pair_source,
                batch_size=256,
                selected_pair_rows=selected_rows,
                pair_diag=pair_diag,
            )

            row_stability = {
                "seed": seed,
                "model": tag,
                "factor_j": factor_j,
                "factor_name": FACTOR_NAMES[factor_j],
                "pair_source": pair_source,
                **experiment_meta,
                **ci,
            }

            rows_stability.append(rows_stability)
            seed_stability_rows.append(rows_stability)

            pair_row = {
                "seed": seed,
                "model": tag,
                "factor_j": factor_j,
                "factor_name": FACTOR_NAMES[factor_j],
                "pair_source": pair_source,
                "n_pairs_found": len(selected_pairs),
                "protected_factors": str(PROTECTED_FACTORS[factor_j]),
                **experiment_meta,
                **pair_diag,
            }

            seed_pair_rows_model.append(pair_row)

        pd.DataFrame(seed_stability_rows).to_csv(
            os.path.join(run_dirs["matched_pairs"], f"matched_pair_stability_{tag}_seed{seed}.csv"),
            index=False
        )

        pd.DataFrame(seed_pair_rows_model).to_csv(
            os.path.join(run_dirs["matched_pairs"], f"pair_search_{tag}_seed{seed}.csv"),
            index=False
        )

        Z_tr_z, Y_tr_z = collect_z_y(predictor, train_loader)
        Z_te_z, Y_te_z = collect_z_y(predictor, test_loader)

        R_tr, Y_tr_r = collect_setrepr_y(predictor, train_loader)
        R_te, Y_te_r = collect_setrepr_y(predictor, test_loader)

        r2_set_repr = linear_probe_r2_train_test(R_tr, Y_tr_r, R_te, Y_te_r)
        r2_z = linear_probe_r2_train_test(Z_tr_z, Y_tr_z, Z_te_z, Y_te_z)

        row_r2 = {
            "seed": seed,
            "model": tag,
            "r2_set_repr_mean": float(np.mean(r2_set_repr)),
            "r2_z_mean": float(np.mean(r2_z)),
            "r2_set_repr_pf": r2_set_repr.tolist(),
            "r2_z_pf": r2_z.tolist(),
            **experiment_meta,
        }

        rows_r2.append(row_r2)

        pd.DataFrame([row_r2]).to_csv(
            os.path.join(run_dirs["identifiability"], f"identifiability_{tag}_seed{seed}.csv"),
            index=False
        )

        mean_attn = collect_factor_query_attention(predictor, test_loader)

        if mean_attn is not None:
            df_attn = pd.DataFrame(
                mean_attn,
                columns=[f"slot_{k}" for k in range(mean_attn.shape[1])]
            )

            df_attn.insert(0, "factor", FACTOR_NAMES[:mean_attn.shape[0]])
            df_attn.insert(0, "loss_tag", loss_tag)
            df_attn.insert(0, "loss_mode", loss_mode)
            df_attn.insert(0, "architecture_tag", arch_tag)
            df_attn.insert(0, "architecture", architecture)
            df_attn.insert(0, "run_id", run_id)
            df_attn.insert(0, "experiment_name", experiment_name)
            df_attn.insert(0, "model", tag)
            df_attn.insert(0, "seed", seed)

            df_attn.to_csv(
                os.path.join(run_dirs["attention"], f"factor_query_attention_{tag}_seed{seed}.csv"),
                index=False
            )

            rows_factor_attention.extend(df_attn.to_dict("records"))

        del predictor
        torch.cuda.empty_cache()


SEED: 45
Sizes | train=226800 | val_sel=12600 | val_cal=12600 | test=25200

--------------------------------------------------------------------------------
BUILDING PAIR CACHE | seed=45 | factor=0

MATCHED PAIR CONSTRUCTION DIAGNOSTICS
factor_j: 0
protected_factors: [1, 2, 3, 4, 5, 6, 7, 8, 9]
raw candidates considered: 16976453
pass target gap: 11998302
pass protected max drift: 0
pass joint protected distance: 0
pass image/slot/mask checks: 0
selected final pairs: 0
note: Raw candidates are candidates examined under capped kNN search, not all possible dataset pairs.
diagnostics: {'factor_j': 0, 'factor_name': 'f0', 'protected_factors': [1, 2, 3, 4, 5, 6, 7, 8, 9], 'n_raw_candidates_considered': 16976453, 'n_pass_target_gap': 11998302, 'n_pass_protected_max_drift': 0, 'n_pass_joint_protected_dist': 0, 'candidate_search_note': 'Raw candidates are candidates examined under capped kNN search, not all possible dataset pairs.', 'n_passing_image_slot_mask_checks': 0, 'n_finally_selected':

screen candidates: 100%|██████████| 3/3 [00:00<00:00, 428.22it/s]



MATCHED PAIR CONSTRUCTION DIAGNOSTICS
factor_j: 1
protected_factors: [0, 2, 3, 4, 5, 6, 7, 8, 9]
raw candidates considered: 17127322
pass target gap: 12006749
pass protected max drift: 3
pass joint protected distance: 3
pass image/slot/mask checks: 1
selected final pairs: 1
note: Raw candidates are candidates examined under capped kNN search, not all possible dataset pairs.
diagnostics: {'factor_j': 1, 'factor_name': 'f1', 'protected_factors': [0, 2, 3, 4, 5, 6, 7, 8, 9], 'n_raw_candidates_considered': 17127322, 'n_pass_target_gap': 12006749, 'n_pass_protected_max_drift': 3, 'n_pass_joint_protected_dist': 3, 'candidate_search_note': 'Raw candidates are candidates examined under capped kNN search, not all possible dataset pairs.', 'n_passing_image_slot_mask_checks': 1, 'n_finally_selected': 1, 'threshold_min_target_gap_std': 0.5, 'threshold_max_protected_drift_std': 0.25, 'threshold_joint_protected_dist_std': 0.6, 'threshold_candidate_knn_k': 1024, 'threshold_max_candidates': 5000, 'th

screen candidates: 100%|██████████| 3/3 [00:00<00:00, 454.21it/s]



MATCHED PAIR CONSTRUCTION DIAGNOSTICS
factor_j: 2
protected_factors: [0, 1, 3, 4, 5, 6, 7, 8, 9]
raw candidates considered: 16822271
pass target gap: 12351693
pass protected max drift: 3
pass joint protected distance: 3
pass image/slot/mask checks: 1
selected final pairs: 1
note: Raw candidates are candidates examined under capped kNN search, not all possible dataset pairs.
diagnostics: {'factor_j': 2, 'factor_name': 'f2', 'protected_factors': [0, 1, 3, 4, 5, 6, 7, 8, 9], 'n_raw_candidates_considered': 16822271, 'n_pass_target_gap': 12351693, 'n_pass_protected_max_drift': 3, 'n_pass_joint_protected_dist': 3, 'candidate_search_note': 'Raw candidates are candidates examined under capped kNN search, not all possible dataset pairs.', 'n_passing_image_slot_mask_checks': 1, 'n_finally_selected': 1, 'threshold_min_target_gap_std': 0.5, 'threshold_max_protected_drift_std': 0.25, 'threshold_joint_protected_dist_std': 0.6, 'threshold_candidate_knn_k': 1024, 'threshold_max_candidates': 5000, 'th

...

collect OOD (mu, var): 100%|██████████| 50/50 [00:17<00:00,  2.92it/s]


[FlatMLP_RESVAR] seed=45 | OOD corruption=gaussian_noise severity=5


collect OOD (mu, var): 100%|██████████| 50/50 [00:17<00:00,  2.79it/s]


Saved: /content/drive/MyDrive/IUThesis/Implementation/Thesis_results/6A/logs/by_architecture/FlatMLP/RESVAR/seed_45/plots/per_factor_coverage_FlatMLP_RESVAR_seed45.png
Saved: /content/drive/MyDrive/IUThesis/Implementation/Thesis_results/6A/logs/by_architecture/FlatMLP/RESVAR/seed_45/plots/per_factor_pit_FlatMLP_RESVAR_seed45_FlatMLP_RESVAR.png
Saved: /content/drive/MyDrive/IUThesis/Implementation/Thesis_results/6A/logs/by_architecture/FlatMLP/RESVAR/seed_45/plots/per_factor_pit_FlatMLP_RESVAR_seed45_FlatMLP_RESVAR+posthocT.png
Saved: /content/drive/MyDrive/IUThesis/Implementation/Thesis_results/6A/logs/by_architecture/FlatMLP/RESVAR/seed_45/plots/per_factor_pit_FlatMLP_RESVAR_seed45_FlatMLP_RESVAR+pfT.png


collect (set_repr, y): 100%|██████████| 50/50 [00:00<00:00, 99.80it/s] 


# **SAVE FINAL AGGREGATED RESULTS**

In [ ]:
# AGGREGATION AND CSV EXPORT

def safe_create_df(list_of_dicts, required_keys=None):
    """
    Creates a DataFrame from a list of dicts.
    Filters out invalid entries or missing required keys.
    """
    if required_keys is None:
        required_keys = []
    filtered = [
        r for r in list_of_dicts
        if isinstance(r, dict) and all(k in r for k in required_keys)
    ]
    if len(filtered) == 0:
        return pd.DataFrame()
    return pd.DataFrame(filtered)

# Filter valid dictionaries for all lists
df_metrics = safe_create_df(rows, required_keys=["model"])
df_conf = safe_create_df(rows_conf, required_keys=["model"])
df_ood_slices = safe_create_df(rows_ood, required_keys=["model"])
df_ood_shift = safe_create_df(rows_ood_shift, required_keys=["model"])
df_stability = safe_create_df(rows_stability, required_keys=["model", "factor_j"])
df_r2 = safe_create_df(rows_r2, required_keys=["model"])
df_pair_search = safe_create_df(rows_pair_search, required_keys=["model"])
df_factor_attention = safe_create_df(rows_factor_attention, required_keys=["model"])

# Save CSVs safely
df_metrics.to_csv(os.path.join(LOG_ROOT, "all_metrics.csv"), index=False)
df_conf.to_csv(os.path.join(LOG_ROOT, "all_conformal.csv"), index=False)
df_ood_slices.to_csv(os.path.join(LOG_ROOT, "all_ood_slices.csv"), index=False)
df_ood_shift.to_csv(os.path.join(LOG_ROOT, "all_ood_shift.csv"), index=False)
df_stability.to_csv(os.path.join(LOG_ROOT, "all_matched_pair_stability.csv"), index=False)
df_r2.to_csv(os.path.join(LOG_ROOT, "all_identifiability.csv"), index=False)
df_pair_search.to_csv(os.path.join(LOG_ROOT, "all_pair_search.csv"), index=False)
df_factor_attention.to_csv(os.path.join(LOG_ROOT, "all_factor_query_attention.csv"), index=False)

# Compute summaries only if the DataFrames are not empty
if not df_metrics.empty:
    df_metrics_ci = add_ci_table(df_metrics)
    df_metrics_ci.to_csv(os.path.join(LOG_ROOT, "summary_metrics_ci.csv"), index=False)
else:
    df_metrics_ci = pd.DataFrame()

if not df_stability.empty:
    df_stability_summary = aggregate_invariance_table(df_stability)
    df_stability_summary.to_csv(os.path.join(LOG_ROOT, "summary_matched_pair_stability_ci.csv"), index=False)
else:
    df_stability_summary = pd.DataFrame()

if not df_r2.empty:
    df_r2_ci = add_ci_table(df_r2, metric_cols=("r2_set_repr_mean", "r2_z_mean"))
    df_r2_ci.to_csv(os.path.join(LOG_ROOT, "summary_identifiability_ci.csv"), index=False)
else:
    df_r2_ci = pd.DataFrame()

print("Saved final aggregated results to:", LOG_ROOT)

# Display if not empty (commented out for thesis)
if not df_metrics_ci.empty:
    pass
if not df_stability_summary.empty:
    pass
if not df_r2_ci.empty:
    pass

Saved final aggregated results to: /content/drive/MyDrive/IUThesis/Implementation/Thesis_results/6A/logs
